In [ ]:
# ════════════════════════════════════════════════════════════════
#  SETUP CELL — 런타임 초기화 후 이 셀만 먼저 실행하면
#  이후 모든 실험 셀을 독립적으로 실행 가능
# ════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd, scipy.io as sio, warnings, copy
from collections import defaultdict
warnings.filterwarnings('ignore')

# ── Drive 마운트 ──────────────────────────────────────────────────────────
#from google.colab import drive
#drive.mount('/content/drive', force_remount=False)

# ── 데이터 로드 ───────────────────────────────────────────────────────────
MAT_PATH   = '/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat'
mat        = sio.loadmat(MAT_PATH, simplify_cells=True)
passes     = mat['mill']

N_FULL     = 9000
FS         = 250          # Hz
SIG_COLS_6 = ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
SIG_COLS_4 = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']   # 채택 구성

# ── 손상 pass 제거 ────────────────────────────────────────────────────────
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({
        'case': c, 'run': r, 'VB': p['VB'],
        'DOC': float(p['DOC']), 'feed': float(p['feed']),
        'material': int(p['material']), 'p': p
    })

# ── VB 보간 (Run1+Interp) ─────────────────────────────────────────────────
df_vb = pd.DataFrame([{'case': x['case'], 'run': x['run'], 'VB': x['VB']}
                       for x in all_passes])
df_vb.loc[(df_vb['run'] == 1) & df_vb['VB'].isna(), 'VB'] = 0.0

def _interp_case(grp):
    grp = grp.sort_values('run').copy()
    grp['VB'] = grp['VB'].interpolate(method='index', limit_area='inside')
    return grp
df_vb = df_vb.groupby('case', group_keys=False).apply(_interp_case)

pass_dict    = {(x['case'], x['run']): x for x in all_passes}
valid_passes = []
for _, row in df_vb.iterrows():
    if np.isnan(row['VB']):
        continue
    item = pass_dict.get((int(row['case']), int(row['run'])))
    if item is None:
        continue
    valid_passes.append({**item, 'VB': row['VB']})

# ── base_run (케이스 내 첫 run) ───────────────────────────────────────────
base_run = {}
for item in valid_passes:
    c = item['case']
    if c not in base_run or item['run'] < base_run[c]['run']:
        base_run[c] = item

# ── 피처 추출 함수 ────────────────────────────────────────────────────────
def extract_features(p, n=N_FULL, sig_cols=SIG_COLS_4):
    """각 신호의 mean/std/rms/peak 반환. (기본: 4sig×4stat = 16-dim)"""
    feats = []
    for col in sig_cols:
        s = p[col][:n].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

def build_X_y(ratio=1.0, sig_cols=SIG_COLS_4, feat_set='raw+delta+meta'):
    """
    ratio       : 0.1~1.0, 신호 앞부분 사용 비율
    sig_cols    : 사용할 신호 채널 목록
    feat_set    : 'raw' | 'delta' | 'raw+delta' | 'raw+delta+meta'
    반환        : X (n_samples, n_features), y (n_samples,), case_arr (n_samples,)
    """
    n = int(N_FULL * ratio)
    X_list, y_list, case_list = [], [], []
    for item in valid_passes:
        c      = item['case']
        f_raw  = extract_features(item['p'], n, sig_cols)
        f_base = extract_features(base_run[c]['p'], n, sig_cols)
        f_delt = f_raw - f_base
        meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
        if feat_set == 'raw':
            vec = f_raw
        elif feat_set == 'delta':
            vec = f_delt
        elif feat_set == 'raw+delta':
            vec = np.concatenate([f_raw, f_delt])
        else:  # raw+delta+meta
            vec = np.concatenate([f_raw, f_delt, meta])
        X_list.append(vec)
        y_list.append(item['VB'])
        case_list.append(c)
    return np.array(X_list), np.array(y_list), np.array(case_list)

# ── GRU용 케이스별 CumDelta 시퀀스 ──────────────────────────────────────
case_items = defaultdict(list)
for item in valid_passes:
    case_items[item['case']].append(item)

case_seqs_delta = {}
for c, items in case_items.items():
    items_s = sorted(items, key=lambda x: x['run'])
    X_raw   = np.array([extract_features(it['p']) for it in items_s])  # (T, 16)
    X_delta = X_raw - X_raw[0]
    y       = np.array([it['VB'] for it in items_s], dtype=float)
    case_seqs_delta[c] = (X_delta, y)

CASES = sorted(case_seqs_delta.keys())

# ── 확인 ─────────────────────────────────────────────────────────────────
print("=" * 52)
print("  SETUP 완료")
print("=" * 52)
print(f"  전체 pass     : {len(all_passes)}")
print(f"  유효 pass     : {len(valid_passes)}  (Run1+Interp VB)")
print(f"  케이스 수     : {len(CASES)}")
print(f"  기본 신호     : {SIG_COLS_4}")
print(f"  피처 차원     : {len(SIG_COLS_4)*4}-dim  (4stat × 4sig)")
print()
print("  정의된 변수/함수:")
print("    valid_passes, base_run, all_passes, passes")
print("    extract_features(p, n, sig_cols)")
print("    build_X_y(ratio, sig_cols, feat_set)")
print("    case_seqs_delta  (GRU용 CumDelta 시퀀스)")
print("    CASES, N_FULL, FS, SIG_COLS_4, SIG_COLS_6")
print()
print("  이 셀 실행 후 아래 실험 셀 어느 것이든 독립 실행 가능")


  SETUP 완료
  전체 pass     : 165
  유효 pass     : 164  (Run1+Interp VB)
  케이스 수     : 16
  기본 신호     : ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
  피처 차원     : 16-dim  (4stat × 4sig)

  정의된 변수/함수:
    valid_passes, base_run, all_passes, passes
    extract_features(p, n, sig_cols)
    build_X_y(ratio, sig_cols, feat_set)
    case_seqs_delta  (GRU용 CumDelta 시퀀스)
    CASES, N_FULL, FS, SIG_COLS_4, SIG_COLS_6

  이 셀 실행 후 아래 실험 셀 어느 것이든 독립 실행 가능


## Preprocessing Techniques Used in the Notebook

This notebook employs a comprehensive set of preprocessing steps to prepare the `mill.mat` dataset for various modeling tasks. The key techniques can be categorized as follows:

### 1. Data Loading and Initial Filtering
- **MAT File Loading**: The `mill.mat` file is loaded using `scipy.io.loadmat`, simplifying the structure for easier access.
- **Damaged Pass Removal**: Specific cases (`(c == 2 and r == 1)` and `(c == 12 and r == 1)`) identified as damaged are removed from the dataset.
- **VB Interpolation**: Missing VB (tool wear) values, particularly for `run=1` where VB is often `NaN`, are handled:
    - `run=1` `NaN` values are initially set to `0.0`.
    - For each `case`, `VB` values are linearly interpolated based on `run` numbers (`method='index', limit_area='inside'`) to fill any remaining `NaN`s within a series of runs.

### 2. Feature Extraction
Several versions of feature extraction functions (`extract_features`, `extract_features_v2`, `extract_features_v3`, `extract_features_v4`) are used, applying statistical measures to signal columns (`smcAC`, `smcDC`, `vib_table`, `vib_spindle`, `AE_table`, `AE_spindle`) over a specified length of the signal (`n=N_FULL * ratio`).

**Common Statistics Applied**:
- `mean()`: Mean of the signal.
- `std()`: Standard deviation of the signal.
- `np.sqrt((s**2).mean())`: Root Mean Square (RMS) of the signal.
- `np.abs(s).max()`: Peak absolute value of the signal.

**Additional Statistics (in `extract_features_v2` and `v3`)**:
- `scipy.stats.kurtosis(s, fisher=True)`: Kurtosis of the signal (Fisher's definition).
- `scipy.stats.skew(s)`: Skewness of the signal.
- `(freqs * fft_mag).sum() / denom`: Spectral centroid of the signal.

**Feature Types**:
- **Raw Features**: Directly extracted statistics from the signal (`f_raw`).
- **Delta Features**: The difference between the current pass's raw features and the `base_run`'s raw features (`f_raw - f_base`). The `base_run` is typically the first run of a given case.
- **Meta Features**: Process parameters such as `DOC` (Depth of Cut), `feed`, and `material` type are included as additional features.
- **Combined Feature Sets**: Features are often concatenated into `raw+delta`, `raw+delta+meta`, `delta-only`, or `delta+meta` combinations.

### 3. Data Scaling
- **Standard Scaling**: Prior to training machine learning models, `sklearn.preprocessing.StandardScaler` is used to standardize features by removing the mean and scaling to unit variance. This is applied within each cross-validation fold to prevent data leakage.

### 4. Signal Segmentation (for specific experiments)
Segmentation techniques are applied to identify and isolate specific phases of the milling process within the raw signal. This allows for feature extraction from more relevant or stable parts of the signal.

- **Two-Stage Unsupervised Segmentation (`segment_pass`, `segment_pass_v2`)**:
    - **Stage 1 (No-load rejection)**: Identifies the end of the no-load region by analyzing windowed RMS features. A baseline RMS (`wf`) and noise floor (`eps`) are calculated from the initial windows, and the no-load region ends when the windowed RMS deviates significantly (`delta * eps`) from the baseline.
    - **Stage 2 (Steady-cut extraction)**: After the no-load region, a fixed-length window (`STEADY_LEN`) slides through the active signal, and the position with the minimum standard deviation is chosen as the start of the steady-cut region. This aims to capture the most stable part of the cutting process.

- **Segment-based Feature Extraction**: Features are extracted from:
    - **Active Region**: The signal after the no-load phase, truncated by a `ratio`.
    - **Steady Region Only**: Features are extracted exclusively from the identified steady-cut segment.
    - **Entry + Steady Region**: Features are extracted from the combined entry and steady-cut segments.

### 5. Sequence Data Preparation (for RNN/GRU models)
- **Time Series Structuring**: For Recurrent Neural Networks (RNNs) and Gated Recurrent Units (GRUs), data is structured as sequences of features per `case` across different `run`s.
- **Delta + Meta Sequencing**: For `GRU-DeltaMeta` models, features are calculated as `delta` relative to the first run, and `meta` (DOC, feed, material) are appended to each time step, forming a `(T, D)` sequence per case, where `T` is the number of runs and `D` is the feature dimension.
- **Padding**: When training in batches, `torch.nn.utils.rnn.pad_sequence` is used to handle variable-length sequences by padding shorter sequences with `NaN`s, which are then masked during loss calculation.
- **Per-Case Scaling**: For sequence models, `StandardScaler` is fitted on the training cases and then used to transform features for both training and testing cases within each LOCO-CV fold.

In [ ]:
import os, glob
results = glob.glob('/content/**/*nasa_milling*', recursive=True) + \
          glob.glob('/root/**/*nasa_milling*', recursive=True) + \
          glob.glob('/tmp/**/*nasa_milling*', recursive=True)
if results:
    for r in results:
        print(r)
else:
    print("dataset_nasa_milling.zip 없음")

In [ ]:
import os

dataset_path = '/content/drive/MyDrive/dataset/nasa_milling'

if os.path.exists(dataset_path):
    print(f"Listing contents of {dataset_path}:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            size = os.path.getsize(os.path.join(root, f))
            print(f"{indent}  {f}  ({size:,} bytes)")
else:
    print(f"Dataset directory not found at {dataset_path}. Please ensure it exists or upload the zip file first.")

In [ ]:
import scipy.io as sio
import numpy as np

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)

# 최상위 키 확인
print("최상위 키:", [k for k in mat.keys() if not k.startswith('_')])

# 주요 변수 구조 탐색
for key in [k for k in mat.keys() if not k.startswith('_')]:
    val = mat[key]
    print(f"\n[{key}]  type={type(val).__name__}", end='')
    if isinstance(val, np.ndarray):
        print(f"  shape={val.shape}  dtype={val.dtype}")
        if val.dtype.names:
            print(f"  fields={val.dtype.names}")
        if val.ndim >= 1 and val.shape[0] > 0:
            sample = val.flat[0]
            if hasattr(sample, 'dtype') and sample.dtype.names:
                print(f"  element fields={sample.dtype.names}")
    else:
        print(f"  value={val}")

In [ ]:
import scipy.io as sio
import numpy as np
import pandas as pd
import os

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

output_dir = '/content/drive/MyDrive/dataset/nasa_milling/csv_passes'
os.makedirs(output_dir, exist_ok=True)

signal_cols = ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
meta_cols   = ['case', 'run', 'VB', 'time', 'DOC', 'feed', 'material']

for p in passes:
    case = int(p['case'])
    run  = int(p['run'])

    df = pd.DataFrame({col: p[col] for col in signal_cols})
    for col in meta_cols:
        df[col] = p[col]

    fname = f'case{case:02d}_run{run:02d}.csv'
    df.to_csv(os.path.join(output_dir, fname), index=False)

print(f"완료: {len(passes)}개 CSV → {output_dir}")
print("샘플 파일 목록 (처음 5개):")
for f in sorted(os.listdir(output_dir))[:5]:
    path = os.path.join(output_dir, f)
    rows = sum(1 for _ in open(path)) - 1
    print(f"  {f}  ({rows:,} rows)")

In [ ]:
import subprocess
result = subprocess.run(['pip', 'install', '-q', 'pdfminer.six'], capture_output=True)

from pdfminer.high_level import extract_text
text = extract_text('/content/drive/MyDrive/dataset/nasa_milling/mill/Readme.pdf')
print(text)

In [ ]:
import scipy.io as sio
import numpy as np
import pandas as pd

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

# --- VB 결측 분포 ---
records = []
for p in passes:
    records.append({
        'case': int(p['case']), 'run': int(p['run']),
        'VB': p['VB'], 'sig_len': len(p['smcAC'])
    })
df_meta = pd.DataFrame(records)

print("=== VB 결측 현황 ===")
nan_df = df_meta[df_meta['VB'].isna()]
print(f"전체 pass: {len(df_meta)},  VB=nan: {len(nan_df)},  유효: {len(df_meta)-len(nan_df)}")
print("\nnan이 있는 케이스별 run 번호:")
for case, grp in nan_df.groupby('case'):
    print(f"  Case {case:02d}: run {sorted(grp['run'].tolist())}")

# --- 신호 길이 동질성 ---
print("\n=== 신호 길이 분포 ===")
print(df_meta['sig_len'].value_counts().sort_index())

# --- 채널별 이상치 (z-score > 5) ---
print("\n=== 채널별 이상치 비율 (|z| > 5) ===")
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
for col in signal_cols:
    all_vals = np.concatenate([p[col] for p in passes])
    mu, sigma = all_vals.mean(), all_vals.std()
    outlier_ratio = (np.abs(all_vals - mu) > 5*sigma).mean()
    print(f"  {col:<15}: {outlier_ratio:.4%}")

In [ ]:
odd = df_meta[df_meta['sig_len'] != 9000]
print("길이 이상 pass:", odd[['case','run','VB','sig_len']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df_valid = df_meta.dropna(subset=['VB'])

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig)

# (1) VB 전체 히스토그램
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df_valid['VB'], bins=30, color='steelblue', edgecolor='white')
for thr, c, lbl in [(0.2,'orange','경미'), (0.5,'red','심각')]:
    ax1.axvline(thr, color=c, linestyle='--', label=f'VB={thr} ({lbl})')
ax1.set_title('VB 전체 분포')
ax1.set_xlabel('VB (mm)'); ax1.set_ylabel('count')
ax1.legend()

# (2) 마모 단계별 샘플 수
bins = [0, 0.2, 0.5, float('inf')]
labels = ['정상(<0.2)', '마모(0.2~0.5)', '심각(>0.5)']
df_valid['wear_stage'] = pd.cut(df_valid['VB'], bins=bins, labels=labels)
stage_cnt = df_valid['wear_stage'].value_counts().reindex(labels)
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(labels, stage_cnt.values, color=['green','orange','red'])
for i, v in enumerate(stage_cnt.values):
    ax2.text(i, v+0.5, str(v), ha='center', fontsize=11)
ax2.set_title('마모 단계별 pass 수')
ax2.set_ylabel('count')

# (3) 케이스별 VB 열화 궤적
ax3 = fig.add_subplot(gs[1, :])
colors = plt.cm.tab20(np.linspace(0, 1, 16))
for i, (case, grp) in enumerate(df_valid.groupby('case')):
    grp_s = grp.sort_values('run')
    ax3.plot(grp_s['run'], grp_s['VB'], marker='o', markersize=4,
             label=f'C{case}', color=colors[i], linewidth=1.5)
ax3.axhline(0.2, color='orange', linestyle='--', linewidth=1)
ax3.axhline(0.5, color='red',    linestyle='--', linewidth=1)
ax3.set_title('케이스별 VB 열화 궤적 (x=run 번호)')
ax3.set_xlabel('run'); ax3.set_ylabel('VB (mm)')
ax3.legend(ncol=8, fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_step2_vb_distribution.png', dpi=150)
plt.show()

print("\n=== VB 기초 통계 ===")
print(df_valid['VB'].describe().round(4))
print("\n마모 단계별 비율:")
print((stage_cnt / len(df_valid) * 100).round(1).to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

# pass별 메타 + VB 최종값(케이스 내 마지막 run의 VB)
records2 = []
for p in passes:
    records2.append({
        'case':int(p['case']),'run':int(p['run']),
        'VB':p['VB'],'time':p['time'],
        'DOC':p['DOC'],'feed':p['feed'],'material':int(p['material'])
    })
df2 = pd.DataFrame(records2)

# 케이스별 최종 VB (마지막 유효 VB)
case_final = df2.dropna(subset=['VB']).groupby('case').apply(
    lambda g: g.loc[g['run'].idxmax(), 'VB']).reset_index()
case_final.columns = ['case','final_VB']
case_meta = df2.drop_duplicates('case')[['case','DOC','feed','material']]
case_final = case_final.merge(case_meta, on='case')
case_final['cond'] = case_final.apply(
    lambda r: f"DOC{r.DOC}_f{r.feed}_m{int(r.material)}", axis=1)

print("=== 케이스별 최종 VB & 조건 ===")
print(case_final[['case','DOC','feed','material','final_VB']].to_string(index=False))

# 재료·DOC·feed별 VB 진행 속도 (run당 평균 VB 증가량)
df_v = df2.dropna(subset=['VB']).sort_values(['case','run'])
df_v['vb_rate'] = df_v.groupby('case')['VB'].diff() / df_v.groupby('case')['run'].diff()
rate_by = df_v.dropna(subset=['vb_rate']).groupby(
    ['material','DOC','feed'])['vb_rate'].mean().reset_index()
rate_by.columns = ['material','DOC','feed','mean_vb_rate']
rate_by['material'] = rate_by['material'].map({1:'cast iron',2:'steel'})
print("\n=== 조건별 평균 마모 속도 (mm/run) ===")
print(rate_by.sort_values('mean_vb_rate', ascending=False).to_string(index=False))

# 동일 조건 반복 케이스 쌍 재현성
pairs = [(1,9),(2,12),(3,11),(4,10),(5,16),(6,15),(7,13),(8,14)]
print("\n=== 동일 조건 반복 케이스 쌍 최종 VB 비교 ===")
for a,b in pairs:
    va = case_final[case_final['case']==a]['final_VB'].values[0]
    vb_ = case_final[case_final['case']==b]['final_VB'].values[0]
    cond = case_final[case_final['case']==a]['cond'].values[0]
    print(f"  Case{a:02d} vs Case{b:02d}  [{cond}]  VB: {va:.3f} vs {vb_:.3f}  diff={abs(va-vb_):.3f}")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, scipy.io as sio
from scipy.stats import spearmanr

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

# pass별 채널 집계 (mean, std, rms)
feat_rows = []
for p in passes:
    row = {'case':int(p['case']),'run':int(p['run']),'VB':p['VB']}
    for col in signal_cols:
        s = p[col][:9000]          # 이상 pass 길이 통일
        row[f'{col}_mean'] = s.mean()
        row[f'{col}_std']  = s.std()
        row[f'{col}_rms']  = np.sqrt((s**2).mean())
    feat_rows.append(row)
df_feat = pd.DataFrame(feat_rows)

# 채널별 기초 통계 (mean 기준)
print("=== pass 단위 채널 mean 기초 통계 ===")
mean_cols = [f'{c}_mean' for c in signal_cols]
print(df_feat[mean_cols].describe().round(4).to_string())

# 채널 간 Pearson 상관 (mean 기준)
corr_mat = df_feat[mean_cols].corr()
print("\n=== 채널 간 Pearson 상관행렬 (mean) ===")
print(corr_mat.round(3).to_string())

fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(corr_mat.values, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(6)); ax.set_yticks(range(6))
labels = signal_cols
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{corr_mat.values[i,j]:.2f}", ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title('Channel Correlation Matrix (pass mean)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_step4_corr.png', dpi=150)
plt.show()

In [ ]:
import numpy as np

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

print("=== pass별 NaN/Inf 검사 ===")
for p in passes:
    c, r = int(p['case']), int(p['run'])
    for col in signal_cols:
        s = p[col][:9000]
        n_nan = np.isnan(s).sum()
        n_inf = np.isinf(s).sum()
        if n_nan > 0 or n_inf > 0:
            print(f"  Case{c:02d} Run{r:02d} [{col}]  nan={n_nan}  inf={n_inf}  max={np.nanmax(np.abs(s)):.3e}")

# 정상 범위 확인: nan/inf 제외한 실제 값 범위
print("\n=== 채널별 유한값 기초 통계 ===")
for col in signal_cols:
    all_vals = np.concatenate([p[col][:9000] for p in passes])
    finite = all_vals[np.isfinite(all_vals)]
    print(f"  {col:<15} n_finite={len(finite):,}  n_bad={len(all_vals)-len(finite):,}  "
          f"min={finite.min():.4f}  max={finite.max():.4f}  mean={finite.mean():.4f}")

In [ ]:
import numpy as np

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
THRESH = 1e6  # 비정상적으로 큰 값 기준

print("=== 비정상 신호값 pass 탐지 (|max| > 1e6) ===")
bad_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    for col in signal_cols:
        s = p[col][:9000]
        if np.abs(s).max() > THRESH:
            bad_passes.append((c, r, col, np.abs(s).max(), np.abs(s).mean()))
            print(f"  Case{c:02d} Run{r:02d} [{col:<15}]  abs_max={np.abs(s).max():.3e}  abs_mean={np.abs(s).mean():.3e}")

print(f"\n총 {len(bad_passes)}건 탐지")

# 정상 pass만의 실제 신호 범위
print("\n=== 정상 pass 채널별 범위 (abs_max < 1e6) ===")
for col in signal_cols:
    vals = []
    for p in passes:
        s = p[col][:9000]
        if np.abs(s).max() < THRESH:
            vals.append(s.mean())
    vals = np.array(vals)
    print(f"  {col:<15} n={len(vals)}  mean_of_means={vals.mean():.4f}  std={vals.std():.4f}  range=[{vals.min():.4f}, {vals.max():.4f}]")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
THRESH = 1e6

# Case02 Run01 제외한 정상 pass 재집계
feat_rows2 = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    s0 = p[signal_cols[0]][:9000]
    if np.abs(s0).max() > THRESH:
        continue
    row = {'case':c,'run':r,'VB':p['VB']}
    for col in signal_cols:
        s = p[col][:9000]
        row[f'{col}_mean'] = s.mean()
        row[f'{col}_std']  = s.std()
        row[f'{col}_rms']  = np.sqrt((s**2).mean())
    feat_rows2.append(row)
df_f2 = pd.DataFrame(feat_rows2)
print(f"분석 대상 pass: {len(df_f2)} (제외: Case02 Run01)")

# 채널 간 상관행렬 (mean)
mean_cols = [f'{c}_mean' for c in signal_cols]
corr = df_f2[mean_cols].corr()
print("\n=== 채널 간 Pearson 상관행렬 (pass mean, 정상 166건) ===")
print(corr.round(3).to_string())

fig, ax = plt.subplots(figsize=(7,5))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(signal_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(signal_cols, fontsize=9)
for i in range(6):
    for j in range(6):
        ax.text(j,i,f"{corr.values[i,j]:.2f}",ha='center',va='center',fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title('Channel Correlation (pass mean, n=166)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_step4_corr_clean.png', dpi=150)
plt.show()

# VB 구간별 box plot
df_vb = df_f2.dropna(subset=['VB']).copy()
df_vb['stage'] = pd.cut(df_vb['VB'], [0,0.2,0.5,2.0], labels=['normal','wear','severe'])

fig, axes = plt.subplots(2,3,figsize=(14,8))
for ax, col in zip(axes.flat, signal_cols):
    data = [df_vb[df_vb['stage']==s][f'{col}_mean'].values for s in ['normal','wear','severe']]
    ax.boxplot(data, labels=['normal','wear','severe'])
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('pass mean')
plt.suptitle('Signal mean by VB stage', y=1.01)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_step4_boxplot.png', dpi=150)
plt.show()

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from scipy.stats import kurtosis as kurt

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
THRESH = 1e6

# 다양한 특징 추출
feat_rows3 = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if np.abs(p['smcAC'][:9000]).max() > THRESH:
        continue
    if np.isnan(p['VB']):
        continue
    row = {'case':c,'run':r,'VB':p['VB']}
    for col in signal_cols:
        s = p[col][:9000]
        row[f'{col}_mean']     = s.mean()
        row[f'{col}_std']      = s.std()
        row[f'{col}_rms']      = np.sqrt((s**2).mean())
        row[f'{col}_kurtosis'] = kurt(s)
        row[f'{col}_peak']     = np.abs(s).max()
    feat_rows3.append(row)
df_f3 = pd.DataFrame(feat_rows3)
print(f"VB 유효 정상 pass: {len(df_f3)}")

feat_types = ['mean','std','rms','kurtosis','peak']
print("\n=== 특징 × 채널별 Spearman 상관계수 (vs VB) ===")
results = []
for ftype in feat_types:
    for col in signal_cols:
        fname = f'{col}_{ftype}'
        rho, pval = spearmanr(df_f3[fname], df_f3['VB'])
        results.append({'feature':fname,'rho':rho,'pval':pval,'abs_rho':abs(rho)})
df_corr = pd.DataFrame(results).sort_values('abs_rho', ascending=False)
print(df_corr[['feature','rho','pval']].head(15).to_string(index=False))
print("\n[하위 5개]")
print(df_corr[['feature','rho','pval']].tail(5).to_string(index=False))

In [ ]:
import numpy as np, matplotlib.pyplot as plt, scipy.io as sio

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

# --- (1) pass 내 파형 (Case01 Run01 정상 케이스) ---
p_sample = next(p for p in passes if int(p['case'])==1 and int(p['run'])==1)
t = np.arange(9000) / 250.0  # 250Hz -> seconds

fig, axes = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
for ax, col in zip(axes, signal_cols):
    ax.plot(t, p_sample[col][:9000], linewidth=0.6)
    ax.set_ylabel(col, fontsize=8)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Case01 Run01 - 6-channel waveform (36s pass)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_step6_waveform.png', dpi=150)
plt.show()

# --- (2) Case01: run 진행에 따른 smcDC 평균 vs VB overlay ---
case1 = [(int(p['run']), p['smcDC'][:9000].mean(), p['VB'])
         for p in passes if int(p['case'])==1
         and np.abs(p['smcDC'][:9000]).max() < 1e6]
case1.sort(key=lambda x: x[0])
runs, dc_means, vbs = zip(*case1)

fig, ax1 = plt.subplots(figsize=(10,4))
ax1.plot(runs, dc_means, 'b-o', markersize=4, label='smcDC mean')
ax1.set_xlabel('run'); ax1.set_ylabel('smcDC mean', color='b')
ax2 = ax1.twinx()
ax2.plot(runs, [v if not (isinstance(v,float) and np.isnan(v)) else None
                for v in vbs], 'r-s', markersize=5, label='VB', linewidth=2)
ax2.set_ylabel('VB (mm)', color='r')
ax1.set_title('Case01: smcDC mean & VB vs run')
fig.legend(loc='upper left', bbox_to_anchor=(0.1,0.9))
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_step6_trend_case01.png', dpi=150)
plt.show()
print("Case01 smcDC means:", [round(v,3) for v in dc_means])
print("Case01 VBs:", [round(v,3) if not np.isnan(v) else 'nan' for v in vbs])

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
THRESH = 1e6

# --- (1) pass 내 구간 패턴: 4등분하여 신호 레벨 변화 확인 ---
p_sample = next(p for p in passes if int(p['case'])==1 and int(p['run'])==1)
n = 9000
print("=== Case01 Run01 구간별 신호 mean (4등분, 각 구간=2250샘플/9초) ===")
for col in signal_cols:
    s = p_sample[col][:n]
    q = [round(s[i*n//4:(i+1)*n//4].mean(), 4) for i in range(4)]
    print(f"  {col:<15}: Q1={q[0]}  Q2={q[1]}  Q3={q[2]}  Q4={q[3]}")

# --- (2) Case01: run별 주요 특징 트렌드 vs VB ---
case1_rows = []
for p in passes:
    if int(p['case']) != 1 or np.abs(p['smcDC'][:9000]).max() > THRESH:
        continue
    row = {'run': int(p['run']), 'VB': p['VB']}
    for col in signal_cols:
        row[f'{col}_mean'] = p[col][:9000].mean()
        row[f'{col}_std']  = p[col][:9000].std()
    case1_rows.append(row)
df_c1 = pd.DataFrame(case1_rows).sort_values('run')
print("\n=== Case01: run별 smcDC_mean / smcAC_std / AE_table_mean / VB ===")
print(df_c1[['run','smcDC_mean','smcAC_std','AE_table_mean','VB']].round(4).to_string(index=False))

# --- (3) 재료별 채널 mean 분포 비교 ---
shift_rows = []
for p in passes:
    if np.abs(p['smcAC'][:9000]).max() > THRESH:
        continue
    row = {'material': int(p['material'])}
    for col in signal_cols:
        row[f'{col}_mean'] = p[col][:9000].mean()
    shift_rows.append(row)
df_shift = pd.DataFrame(shift_rows)
print("\n=== 재료별 채널 mean (cast iron=1 vs steel=2) ===")
for col in signal_cols:
    g1 = df_shift[df_shift['material']==1][f'{col}_mean']
    g2 = df_shift[df_shift['material']==2][f'{col}_mean']
    print(f"  {col:<15}  iron: {g1.mean():.4f}+/-{g1.std():.4f}   steel: {g2.mean():.4f}+/-{g2.std():.4f}")

# --- (4) 케이스별 smcDC_mean (케이스 간 도메인 shift) ---
case_means = []
for p in passes:
    if np.abs(p['smcDC'][:9000]).max() > THRESH:
        continue
    case_means.append({'case': int(p['case']), 'smcDC_mean': p['smcDC'][:9000].mean(),
                       'material': int(p['material'])})
df_cm = pd.DataFrame(case_means).groupby('case').agg(
    smcDC_mean=('smcDC_mean','mean'), material=('material','first')).reset_index()
print("\n=== 케이스별 smcDC_mean 평균 (도메인 shift 확인) ===")
print(df_cm.sort_values('smcDC_mean', ascending=False).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, scipy.io as sio

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']
THRESH = 1e6

# ── (1) pass 내 진입/이탈 구간 시각화: Case01 Run01 smcDC ──────────────
p01 = next(p for p in passes if int(p['case'])==1 and int(p['run'])==1)
s = p01['smcDC'][:9000]
t = np.arange(9000) / 250.0  # 초 단위

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(t, s, linewidth=0.7, color='steelblue')
ax.axvspan(0,    9,   alpha=0.15, color='red',   label='Entry cut (Q1)')
ax.axvspan(9,   27,   alpha=0.15, color='green', label='Steady-state (Q2-Q3)')
ax.axvspan(27,  36,   alpha=0.15, color='orange',label='Exit cut (Q4)')
ax.axhline(s[2250:6750].mean(), linestyle='--', color='green',
           linewidth=1.2, label=f'Steady mean={s[2250:6750].mean():.3f}')
ax.axhline(s.mean(), linestyle='--', color='gray',
           linewidth=1.0, label=f'Full mean={s.mean():.3f}')
ax.set_xlabel('Time (s)'); ax.set_ylabel('smcDC')
ax.set_title('Case01 Run01 — entry/exit cut regions (smcDC)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_entry_exit.png', dpi=130)
plt.show()

# ── (2) 상대 변화량(delta) feature: Case01 run별 raw vs delta ─────────────
case1_rows = []
for p in passes:
    if int(p['case']) != 1 or np.abs(p['smcDC'][:9000]).max() > THRESH:
        continue
    s = p['smcDC'][:9000]
    case1_rows.append({
        'run': int(p['run']),
        'VB':  p['VB'],
        'raw_full':   s.mean(),                        # 전체 구간 mean
        'raw_steady': s[2250:6750].mean(),             # steady-state only
    })
df_c1 = pd.DataFrame(case1_rows).sort_values('run').reset_index(drop=True)

# delta = 현재 run - run1 (케이스 내 첫 run 기준 상대 변화량)
base_full   = df_c1.loc[0, 'raw_full']
base_steady = df_c1.loc[0, 'raw_steady']
df_c1['delta_full']   = df_c1['raw_full']   - base_full
df_c1['delta_steady'] = df_c1['raw_steady'] - base_steady

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
runs = df_c1['run'].values
vbs  = df_c1['VB'].values

# 왼쪽: raw mean 비교
ax = axes[0]
ax.plot(runs, df_c1['raw_full'],   'b-o', markersize=4, label='full mean')
ax.plot(runs, df_c1['raw_steady'], 'g-s', markersize=4, label='steady mean')
ax2 = ax.twinx()
ax2.plot(runs, vbs, 'r--', linewidth=1.5, label='VB')
ax2.set_ylabel('VB (mm)', color='r')
ax.set_title('Case01 smcDC: raw (full vs steady-state)')
ax.set_xlabel('run'); ax.set_ylabel('smcDC mean')
ax.legend(loc='upper left', fontsize=8)

# 오른쪽: delta (상대 변화량)
ax = axes[1]
ax.plot(runs, df_c1['delta_full'],   'b-o', markersize=4, label='delta_full')
ax.plot(runs, df_c1['delta_steady'], 'g-s', markersize=4, label='delta_steady')
ax2 = ax.twinx()
ax2.plot(runs, vbs, 'r--', linewidth=1.5, label='VB')
ax2.set_ylabel('VB (mm)', color='r')
ax.set_title('Case01 smcDC: delta (relative change from run1)')
ax.set_xlabel('run'); ax.set_ylabel('delta smcDC mean')
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_delta_case01.png', dpi=130)
plt.show()

# ── (3) 다중 케이스 delta 비교: 케이스 초기화 후 얼마나 정렬되는지 ─────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for case_id in range(1, 17):
    rows = []
    for p in passes:
        if int(p['case']) != case_id or np.abs(p['smcDC'][:9000]).max() > THRESH:
            continue
        rows.append({'run': int(p['run']), 'steady': p['smcDC'][2250:6750].mean()})
    if not rows:
        continue
    df_tmp = pd.DataFrame(rows).sort_values('run').reset_index(drop=True)
    base = df_tmp.loc[0, 'steady']
    axes[0].plot(df_tmp['run'], df_tmp['steady'],     marker='.', markersize=3, linewidth=1, alpha=0.7)
    axes[1].plot(df_tmp['run'], df_tmp['steady']-base, marker='.', markersize=3, linewidth=1, alpha=0.7)

axes[0].set_title('All cases: raw smcDC steady mean'); axes[0].set_xlabel('run')
axes[1].set_title('All cases: delta smcDC (run1 기준)'); axes[1].set_xlabel('run')
for ax in axes:
    ax.set_ylabel('smcDC')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/eda_delta_allcases.png', dpi=130)
plt.show()

print("저장 완료: eda_entry_exit.png / eda_delta_case01.png / eda_delta_allcases.png")

# ── 수치 확인 ─────────────────────────────────────────────────────────────
print("\n=== Case01 run별 raw vs delta (smcDC) ===")
print(df_c1[['run','raw_full','raw_steady','delta_full','delta_steady','VB']].round(4).to_string(index=False))

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

THRESH      = 1e6
N_FULL      = 9000
RATIOS      = [r / 10 for r in range(1, 11)]
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
FEAT_NAMES  = [f'{c}_{s}' for c in signal_cols for s in ['mean','std','rms','peak']]  # 24개

def extract_features(p, n):
    feats = []
    for col in signal_cols:
        s = p[col][:n]
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── 1. 전체 pass 로드 (손상 2개 제거) ──────────────────────────────────────
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({'case': c, 'run': r, 'VB': p['VB'], 'p': p})

# ── 2. 케이스별 기준(run 최솟값) 신호 특징 사전 구축 ─────────────────────
base_run = {}  # case -> min run pass
for item in all_passes:
    c = item['case']
    if c not in base_run or item['run'] < base_run[c]['run']:
        base_run[c] = item

# ── 3. 유효 pass (VB not nan) ─────────────────────────────────────────────
valid_passes = [item for item in all_passes if not np.isnan(item['VB'])]
cases = sorted(set(item['case'] for item in valid_passes))
print(f"유효 pass: {len(valid_passes)} / 케이스: {len(cases)}")

# ── 4. 모델 정의 ──────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

# ── 5. LOCO CV ────────────────────────────────────────────────────────────
results = []

for ratio in RATIOS:
    n = int(N_FULL * ratio)

    # 비율별 특징 행렬 구성
    X_raw, X_delta, y_arr, case_arr = [], [], [], []
    for item in valid_passes:
        c = item['case']
        f_raw  = extract_features(item['p'], n)
        f_base = extract_features(base_run[c]['p'], n)
        f_comb = np.concatenate([f_raw, f_raw - f_base])   # 48-dim
        X_raw.append(f_raw)
        X_delta.append(f_comb)
        y_arr.append(item['VB'])
        case_arr.append(c)

    X_raw   = np.array(X_raw)
    X_delta = np.array(X_delta)
    y_arr   = np.array(y_arr)
    case_arr = np.array(case_arr)

    for mname, mproto in MODELS.items():
        rmses_raw, rmses_delta = [], []
        for tc in cases:
            tr = case_arr != tc
            te = case_arr == tc
            if te.sum() == 0:
                continue
            for X, rmse_list in [(X_raw, rmses_raw), (X_delta, rmses_delta)]:
                sc = StandardScaler()
                Xtr = sc.fit_transform(X[tr]);  Xte = sc.transform(X[te])
                m = copy.deepcopy(mproto)
                m.fit(Xtr, y_arr[tr])
                pred = m.predict(Xte)
                rmse_list.append(np.sqrt(mean_squared_error(y_arr[te], pred)))

        results.append({'ratio': ratio, 'model': mname,
                        'rmse_raw':   np.mean(rmses_raw),
                        'rmse_delta': np.mean(rmses_delta)})

    print(f"  ratio={ratio:.1f}  done")

# ── 6. 결과 정리 ──────────────────────────────────────────────────────────
df_res = pd.DataFrame(results)

print("\n==============================")
print("  LOCO-CV RMSE  [Raw features]")
print("==============================")
print(df_res.pivot(index='ratio', columns='model', values='rmse_raw').round(4).to_string())

print("\n======================================")
print("  LOCO-CV RMSE  [Raw + Delta features]")
print("======================================")
print(df_res.pivot(index='ratio', columns='model', values='rmse_delta').round(4).to_string())

print("\n=== 모델별 최적 입력 비율 (Raw) ===")
for mname in MODELS:
    sub  = df_res[df_res['model'] == mname]
    best = sub.loc[sub['rmse_raw'].idxmin()]
    print(f"  {mname:<6}  ratio={best['ratio']:.1f}  RMSE={best['rmse_raw']:.4f}")

print("\n=== 모델별 최적 입력 비율 (Raw+Delta) ===")
for mname in MODELS:
    sub  = df_res[df_res['model'] == mname]
    best = sub.loc[sub['rmse_delta'].idxmin()]
    print(f"  {mname:<6}  ratio={best['ratio']:.1f}  RMSE={best['rmse_delta']:.4f}")

# ── 7. 결과 플롯 (파일 저장, 인라인 표시 없음) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}
x = [r * 100 for r in RATIOS]

for ax, feat_col, title in [
    (axes[0], 'rmse_raw',   'Raw features'),
    (axes[1], 'rmse_delta', 'Raw + Delta features'),
]:
    for mname in MODELS:
        sub = df_res[df_res['model'] == mname].sort_values('ratio')
        ax.plot(x, sub[feat_col].values, marker='o', markersize=5,
                label=mname, color=colors[mname], linewidth=1.8)
    ax.axvline(25, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='entry cut end (~25%)')
    ax.set_xlabel('Input length (%)')
    ax.set_ylabel('RMSE (mm)')
    ax.set_title(f'LOCO-CV RMSE vs Input ratio\n[{title}]')
    ax.legend(fontsize=9)
    ax.set_xticks(x)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/exp_input_ratio_rmse.png', dpi=150)
plt.close()
print("\n플롯 저장 완료: /content/drive/MyDrive/dataset/nasa_milling/exp_input_ratio_rmse.png")

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

THRESH      = 1e6
N_FULL      = 9000
RATIOS      = [r / 10 for r in range(1, 11)]
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
EPS         = 1e-8

def extract_features(p, n):
    feats = []
    for col in signal_cols:
        s = p[col][:n]
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── 전체 pass 로드 (손상 2개 제거) ──────────────────────────────────────
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({
        'case': c, 'run': r, 'VB': p['VB'],
        'DOC': float(p['DOC']), 'feed': float(p['feed']),
        'material': int(p['material']), 'p': p
    })

# 케이스별 run 목록 인덱스 (run-to-run 이전 run 조회용)
case_run_index = {}   # (case, run) -> item
for item in all_passes:
    case_run_index[(item['case'], item['run'])] = item

valid_passes = [item for item in all_passes if not np.isnan(item['VB'])]
cases = sorted(set(item['case'] for item in valid_passes))
print(f"유효 pass: {len(valid_passes)} / 케이스: {len(cases)}")

MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

def loco_cv(X_dict, y_arr, case_arr, cases, models):
    """X_dict: {'method_name': X_array}, return DataFrame of results"""
    rows = []
    for mname, mproto in models.items():
        for feat_name, X in X_dict.items():
            rmses = []
            for tc in cases:
                tr = case_arr != tc
                te = case_arr == tc
                if te.sum() == 0:
                    continue
                sc = StandardScaler()
                Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
                m = copy.deepcopy(mproto)
                m.fit(Xtr, y_arr[tr])
                pred = m.predict(Xte)
                rmses.append(np.sqrt(mean_squared_error(y_arr[te], pred)))
            rows.append({'model': mname, 'feat': feat_name, 'rmse': np.mean(rmses)})
    return pd.DataFrame(rows)

# ════════════════════════════════════════════════════
# 실험 A: 조건 메타데이터 추가  (raw 24 + DOC/feed/material 3 = 27-dim)
# ════════════════════════════════════════════════════
print("\n[실험 A] 조건 메타데이터 추가")
res_meta = []
for ratio in RATIOS:
    n = int(N_FULL * ratio)
    X_raw, X_meta, y_arr, case_arr = [], [], [], []
    for item in valid_passes:
        f = extract_features(item['p'], n)
        X_raw.append(f)
        X_meta.append(np.append(f, [item['DOC'], item['feed'], item['material']]))
        y_arr.append(item['VB'])
        case_arr.append(item['case'])
    X_raw   = np.array(X_raw)
    X_meta  = np.array(X_meta)
    y_arr   = np.array(y_arr)
    case_arr = np.array(case_arr)

    df = loco_cv({'raw': X_raw, 'meta': X_meta}, y_arr, case_arr, cases, MODELS)
    df['ratio'] = ratio
    res_meta.append(df)
    print(f"  ratio={ratio:.1f}  done")

df_meta = pd.concat(res_meta, ignore_index=True)

# ════════════════════════════════════════════════════
# 실험 B: run-to-run 비율  (features_n / features_{n-1}, run1 제외)
# ════════════════════════════════════════════════════
print("\n[실험 B] run-to-run 비율")

# 유효 pass 중 이전 run이 존재하는 것만 선택
def get_prev_run(item, case_run_index):
    c, r = item['case'], item['run']
    # 직전 run 번호 탐색 (연속적이지 않을 수 있으므로 순차 탐색)
    prev_runs = [k[1] for k in case_run_index if k[0] == c and k[1] < r]
    if not prev_runs:
        return None
    return case_run_index[(c, max(prev_runs))]

res_r2r = []
for ratio in RATIOS:
    n = int(N_FULL * ratio)
    X_ratio, y_arr, case_arr = [], [], []
    for item in valid_passes:
        prev = get_prev_run(item, case_run_index)
        if prev is None:
            continue
        f_cur  = extract_features(item['p'], n)
        f_prev = extract_features(prev['p'], n)
        X_ratio.append(f_cur / (f_prev + EPS))
        y_arr.append(item['VB'])
        case_arr.append(item['case'])

    X_ratio  = np.array(X_ratio)
    y_arr    = np.array(y_arr)
    case_arr = np.array(case_arr)

    df = loco_cv({'r2r': X_ratio}, y_arr, case_arr, cases, MODELS)
    df['ratio'] = ratio
    res_r2r.append(df)
    print(f"  ratio={ratio:.1f}  n_pass={len(y_arr)}  done")

df_r2r = pd.concat(res_r2r, ignore_index=True)

# ════════════════════════════════════════════════════
# 결과 출력
# ════════════════════════════════════════════════════
print("\n" + "="*60)
print("실험 A — LOCO-CV RMSE : Raw features")
print("="*60)
print(df_meta[df_meta['feat']=='raw'].pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

print("\n" + "="*60)
print("실험 A — LOCO-CV RMSE : Raw + Condition metadata")
print("="*60)
print(df_meta[df_meta['feat']=='meta'].pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

print("\n" + "="*60)
print("실험 B — LOCO-CV RMSE : Run-to-run ratio")
print("="*60)
print(df_r2r[df_r2r['feat']=='r2r'].pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

# 모델별 최적 비율 요약
print("\n=== 모델별 최적 입력 비율 요약 ===")
print(f"{'방법':<25} {'모델':<8} {'best ratio':>12} {'RMSE':>8}")
print("-"*55)
for df_exp, fname, label in [
    (df_meta[df_meta['feat']=='raw'],  'raw',  'Raw'),
    (df_meta[df_meta['feat']=='meta'], 'meta', 'Raw+Meta'),
    (df_r2r [df_r2r ['feat']=='r2r'],  'r2r',  'Run-to-run ratio'),
]:
    for mname in MODELS:
        sub  = df_exp[df_exp['model'] == mname]
        best = sub.loc[sub['rmse'].idxmin()]
        print(f"  {label:<23} {mname:<8} {best['ratio']:>10.1f}   {best['rmse']:>7.4f}")

# ── 플롯 저장 ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}
x = [r * 100 for r in RATIOS]

for ax, (df_exp, feat_key, title) in zip(axes, [
    (df_meta, 'raw',  'Raw features'),
    (df_meta, 'meta', 'Raw + Condition metadata'),
    (df_r2r,  'r2r',  'Run-to-run ratio'),
]):
    sub_df = df_exp[df_exp['feat'] == feat_key]
    for mname in MODELS:
        sub = sub_df[sub_df['model'] == mname].sort_values('ratio')
        ax.plot(x, sub['rmse'].values, marker='o', markersize=5,
                label=mname, color=colors[mname], linewidth=1.8)
    ax.axvline(25, color='gray', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_title(title); ax.set_xlabel('Input length (%)'); ax.set_ylabel('RMSE (mm)')
    ax.set_xticks(x); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/exp_meta_r2r_rmse.png', dpi=150)
plt.close()
print("\n플롯 저장: /content/drive/MyDrive/dataset/nasa_milling/exp_meta_r2r_rmse.png")

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

THRESH      = 1e6
N_FULL      = 9000
RATIOS      = [r / 10 for r in range(1, 11)]
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

def extract_features(p, n):
    feats = []
    for col in signal_cols:
        s = p[col][:n]
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({'case': c, 'run': r, 'VB': p['VB'], 'p': p})

base_run = {}
for item in all_passes:
    c = item['case']
    if c not in base_run or item['run'] < base_run[c]['run']:
        base_run[c] = item

valid_passes = [item for item in all_passes if not np.isnan(item['VB'])]
cases        = sorted(set(item['case'] for item in valid_passes))

MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

results = []

for ratio in RATIOS:
    n = int(N_FULL * ratio)
    X_delta, y_arr, case_arr = [], [], []
    for item in valid_passes:
        c     = item['case']
        f_raw  = extract_features(item['p'], n)
        f_base = extract_features(base_run[c]['p'], n)
        X_delta.append(f_raw - f_base)   # delta only (24-dim)
        y_arr.append(item['VB'])
        case_arr.append(c)

    X_delta  = np.array(X_delta)
    y_arr    = np.array(y_arr)
    case_arr = np.array(case_arr)

    for mname, mproto in MODELS.items():
        rmses = []
        for tc in cases:
            tr = case_arr != tc
            te = case_arr == tc
            if te.sum() == 0:
                continue
            sc   = StandardScaler()
            Xtr  = sc.fit_transform(X_delta[tr])
            Xte  = sc.transform(X_delta[te])
            m    = copy.deepcopy(mproto)
            m.fit(Xtr, y_arr[tr])
            pred = m.predict(Xte)
            rmses.append(np.sqrt(mean_squared_error(y_arr[te], pred)))
        results.append({'ratio': ratio, 'model': mname, 'rmse': np.mean(rmses)})

    print(f"  ratio={ratio:.1f}  done")

df_res = pd.DataFrame(results)

print("\n" + "="*55)
print("  LOCO-CV RMSE  [Delta-only features (24-dim)]")
print("="*55)
print(df_res.pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

print("\n=== 모델별 최적 입력 비율 ===")
for mname in MODELS:
    sub  = df_res[df_res['model'] == mname]
    best = sub.loc[sub['rmse'].idxmin()]
    print(f"  {mname:<6}  ratio={best['ratio']:.1f}  RMSE={best['rmse']:.4f}")

# ── 이전 실험 결과와 나란히 비교 ─────────────────────────────────────────
print("\n" + "="*65)
print("  전체 방법 비교 (각 모델별 최고 성능)")
print("="*65)
prev = {
    'Raw':            {'Ridge':0.1713,'RF':0.1609,'XGB':0.1612,'SVR':0.1711},
    'Raw+Delta':      {'Ridge':0.1657,'RF':0.1233,'XGB':0.1187,'SVR':0.1268},
    'Raw+Meta':       {'Ridge':0.1278,'RF':0.1613,'XGB':0.1573,'SVR':0.1229},
    'Run-to-run':     {'Ridge':0.2834,'RF':0.2016,'XGB':0.1868,'SVR':0.1977},
}
delta_only_best = {mname: df_res[df_res['model']==mname]['rmse'].min() for mname in MODELS}
prev['Delta-only'] = delta_only_best

df_cmp = pd.DataFrame(prev).T
print(df_cmp.round(4).to_string())

# ── 플롯 저장 ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}
x = [r * 100 for r in RATIOS]
for mname in MODELS:
    sub = df_res[df_res['model'] == mname].sort_values('ratio')
    ax.plot(x, sub['rmse'].values, marker='o', markersize=5,
            label=mname, color=colors[mname], linewidth=1.8)
ax.axvline(25, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='entry cut end (~25%)')
ax.set_title('LOCO-CV RMSE vs Input ratio [Delta-only features]')
ax.set_xlabel('Input length (%)'); ax.set_ylabel('RMSE (mm)')
ax.set_xticks(x); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/exp_delta_only_rmse.png', dpi=150)
plt.close()
print("\n플롯 저장: /content/drive/MyDrive/dataset/nasa_milling/exp_delta_only_rmse.png")

In [ ]:
import scipy.io as sio, numpy as np, pandas as pd

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

# run1의 VB 값 전수 확인
run1s = [(int(p['case']), int(p['run']), p['VB'])
         for p in passes if int(p['run']) == 1]
run1s.sort()

print("=== 케이스별 run1 VB ===")
for c, r, vb in run1s:
    flag = '  ← NaN (측정 안됨)' if (isinstance(vb, float) and np.isnan(vb)) else ''
    print(f"  Case{c:02d}  VB={vb}{flag}")

# delta base로 실제 쓰인 run (run1 NaN이면 다음 run으로 대체된 케이스)
print("\n=== delta base로 사용된 run (케이스 내 최솟값 run) ===")
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({'case': c, 'run': r, 'VB': p['VB']})

base_run = {}
for item in all_passes:
    c = item['case']
    if c not in base_run or item['run'] < base_run[c]['run']:
        base_run[c] = item

for c in sorted(base_run):
    b = base_run[c]
    print(f"  Case{c:02d}  base_run={b['run']}  VB={b['VB']}")

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, copy, warnings
import matplotlib; matplotlib.use('Agg')
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

THRESH      = 1e6
N_FULL      = 9000
RATIOS      = [r / 10 for r in range(1, 11)]
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

def extract_features(p, n):
    feats = []
    for col in signal_cols:
        s = p[col][:n]
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── 전체 pass 로드 (손상 2개 제거) ──────────────────────────────────────
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({'case': c, 'run': r, 'VB': p['VB'], 'p': p})

# ── VB 테이블 구성 후 조건별 보간 ────────────────────────────────────────
df_vb = pd.DataFrame([{'case': x['case'], 'run': x['run'], 'VB': x['VB']}
                       for x in all_passes])

def build_vb_imputed(df, fill_run1=False, interpolate=False):
    df = df.copy()
    if fill_run1:
        # run1 NaN → 0 (단, 신호가 존재하는 pass만)
        mask = (df['run'] == 1) & df['VB'].isna()
        df.loc[mask, 'VB'] = 0.0
    if interpolate:
        # 케이스 내 run 순서로 선형 보간 (경계 외 extrapolation 제외)
        def interp_case(grp):
            grp = grp.sort_values('run').copy()
            grp['VB'] = grp['VB'].interpolate(method='index', limit_area='inside')
            return grp
        df = df.groupby('case', group_keys=False).apply(interp_case)
    return df

df_orig   = build_vb_imputed(df_vb, fill_run1=False, interpolate=False)
df_fill   = build_vb_imputed(df_vb, fill_run1=True,  interpolate=False)
df_interp = build_vb_imputed(df_vb, fill_run1=True,  interpolate=True)

# pass 객체 dict (빠른 조회)
pass_dict = {(x['case'], x['run']): x['p'] for x in all_passes}

def make_valid_passes(df_vb_imp):
    rows = []
    for _, row in df_vb_imp.iterrows():
        if np.isnan(row['VB']):
            continue
        p = pass_dict.get((int(row['case']), int(row['run'])))
        if p is None:
            continue
        rows.append({'case': int(row['case']), 'run': int(row['run']),
                     'VB': row['VB'], 'p': p})
    return rows

cond_passes = {
    'Original':        make_valid_passes(df_orig),
    'Run1-fill':       make_valid_passes(df_fill),
    'Run1+Interp':     make_valid_passes(df_interp),
}
for cname, vp in cond_passes.items():
    print(f"{cname:<15}: {len(vp)} valid passes")

# ── 보간 결과 확인 (Case13·16 예시) ─────────────────────────────────────
print("\n=== Case13 VB 보간 전후 ===")
print(df_interp[df_interp['case']==13][['case','run','VB']].to_string(index=False))
print("\n=== Case16 VB 보간 전후 ===")
print(df_interp[df_interp['case']==16][['case','run','VB']].to_string(index=False))

# ── LOCO-CV (delta-only) ─────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

def run_loco_cv(valid_passes):
    base_run = {}
    for item in valid_passes:
        c = item['case']
        if c not in base_run or item['run'] < base_run[c]['run']:
            base_run[c] = item
    cases = sorted(set(item['case'] for item in valid_passes))
    results = []
    for ratio in RATIOS:
        n = int(N_FULL * ratio)
        X, y_arr, case_arr = [], [], []
        for item in valid_passes:
            c     = item['case']
            f_raw  = extract_features(item['p'], n)
            f_base = extract_features(base_run[c]['p'], n)
            X.append(f_raw - f_base)
            y_arr.append(item['VB'])
            case_arr.append(c)
        X        = np.array(X)
        y_arr    = np.array(y_arr)
        case_arr = np.array(case_arr)
        for mname, mproto in MODELS.items():
            rmses = []
            for tc in cases:
                tr = case_arr != tc
                te = case_arr == tc
                if te.sum() == 0:
                    continue
                sc  = StandardScaler()
                Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
                m   = copy.deepcopy(mproto)
                m.fit(Xtr, y_arr[tr])
                pred = m.predict(Xte)
                rmses.append(np.sqrt(mean_squared_error(y_arr[te], pred)))
            results.append({'ratio': ratio, 'model': mname, 'rmse': np.mean(rmses)})
    return pd.DataFrame(results)

all_results = {}
for cname, vp in cond_passes.items():
    print(f"\n[{cname}] LOCO-CV 실행 중...")
    all_results[cname] = run_loco_cv(vp)
    print(f"  완료")

# ── 결과 출력 ─────────────────────────────────────────────────────────────
for cname, df_res in all_results.items():
    print(f"\n{'='*55}")
    print(f"  Delta-only RMSE  [{cname}]")
    print(f"{'='*55}")
    print(df_res.pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

# ── 조건별 최고 성능 비교 ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  조건별 모델 최고 RMSE 비교 (delta-only)")
print(f"{'='*65}")
summary = {}
for cname, df_res in all_results.items():
    summary[cname] = {mname: df_res[df_res['model']==mname]['rmse'].min()
                      for mname in MODELS}
df_summary = pd.DataFrame(summary).T
print(df_summary.round(4).to_string())

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, copy, warnings
import matplotlib; matplotlib.use('Agg')
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

THRESH      = 1e6
N_FULL      = 9000
RATIOS      = [r / 10 for r in range(1, 11)]
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

def extract_features(p, n):
    feats = []
    for col in signal_cols:
        s = p[col][:n]
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── 전체 pass 로드 ───────────────────────────────────────────────────────
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({'case': c, 'run': r, 'VB': p['VB'],
                       'DOC': float(p['DOC']), 'feed': float(p['feed']),
                       'material': int(p['material']), 'p': p})

# ── Run1+Interp 보간 (이전 실험 최고 조건) ──────────────────────────────
df_vb = pd.DataFrame([{'case': x['case'], 'run': x['run'], 'VB': x['VB']}
                       for x in all_passes])
mask_run1_nan = (df_vb['run'] == 1) & df_vb['VB'].isna()
df_vb.loc[mask_run1_nan, 'VB'] = 0.0

def interp_case(grp):
    grp = grp.sort_values('run').copy()
    grp['VB'] = grp['VB'].interpolate(method='index', limit_area='inside')
    return grp
df_vb = df_vb.groupby('case', group_keys=False).apply(interp_case)

pass_dict = {(x['case'], x['run']): x for x in all_passes}
valid_passes = []
for _, row in df_vb.iterrows():
    if np.isnan(row['VB']):
        continue
    item = pass_dict.get((int(row['case']), int(row['run'])))
    if item is None:
        continue
    valid_passes.append({**item, 'VB': row['VB']})

cases = sorted(set(x['case'] for x in valid_passes))
print(f"유효 pass: {len(valid_passes)} (Run1+Interp)")

# ── base_run (케이스 내 첫 run) ──────────────────────────────────────────
base_run = {}
for item in valid_passes:
    c = item['case']
    if c not in base_run or item['run'] < base_run[c]['run']:
        base_run[c] = item

# ── LOCO-CV ──────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

results = []
for ratio in RATIOS:
    n = int(N_FULL * ratio)

    X_delta, X_delta_meta, X_raw_delta_meta = [], [], []
    y_arr, case_arr = [], []

    for item in valid_passes:
        c      = item['case']
        f_raw  = extract_features(item['p'], n)
        f_base = extract_features(base_run[c]['p'], n)
        f_delt = f_raw - f_base
        meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)

        X_delta.append(f_delt)                                      # 24-dim
        X_delta_meta.append(np.concatenate([f_delt, meta]))         # 27-dim
        X_raw_delta_meta.append(np.concatenate([f_raw, f_delt, meta]))  # 51-dim
        y_arr.append(item['VB'])
        case_arr.append(c)

    feat_sets = {
        'Delta-only':      np.array(X_delta),
        'Delta+Meta':      np.array(X_delta_meta),
        'Raw+Delta+Meta':  np.array(X_raw_delta_meta),
    }
    y_arr    = np.array(y_arr)
    case_arr = np.array(case_arr)

    for fname, X in feat_sets.items():
        for mname, mproto in MODELS.items():
            rmses = []
            for tc in cases:
                tr = case_arr != tc;  te = case_arr == tc
                if te.sum() == 0: continue
                sc  = StandardScaler()
                Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
                m   = copy.deepcopy(mproto)
                m.fit(Xtr, y_arr[tr])
                pred = m.predict(Xte)
                rmses.append(np.sqrt(mean_squared_error(y_arr[te], pred)))
            results.append({'ratio': ratio, 'feat': fname,
                            'model': mname, 'rmse': np.mean(rmses)})
    print(f"  ratio={ratio:.1f}  done")

df_res = pd.DataFrame(results)

# ── 결과 출력 ─────────────────────────────────────────────────────────────
for fname in ['Delta-only', 'Delta+Meta', 'Raw+Delta+Meta']:
    print(f"\n{'='*55}")
    print(f"  LOCO-CV RMSE  [{fname}]")
    print(f"{'='*55}")
    sub = df_res[df_res['feat'] == fname]
    print(sub.pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

# ── 최고 성능 요약 ────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("  feature set별 모델 최고 RMSE 비교")
print(f"{'='*60}")
summary = {}
for fname in ['Delta-only', 'Delta+Meta', 'Raw+Delta+Meta']:
    sub = df_res[df_res['feat'] == fname]
    summary[fname] = {mname: sub[sub['model']==mname]['rmse'].min()
                      for mname in MODELS}
df_sum = pd.DataFrame(summary).T
print(df_sum.round(4).to_string())

print(f"\n전체 최고: {df_res['rmse'].min():.4f}  "
      f"(feat={df_res.loc[df_res['rmse'].idxmin(),'feat']}, "
      f"model={df_res.loc[df_res['rmse'].idxmin(),'model']}, "
      f"ratio={df_res.loc[df_res['rmse'].idxmin(),'ratio']:.1f})")

In [ ]:
import numpy as np, pandas as pd, scipy.io as sio, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

mat = sio.loadmat('/content/drive/MyDrive/dataset/nasa_milling/mill/mill.mat', simplify_cells=True)
passes = mat['mill']

N_FULL      = 9000
RATIO       = 1.0
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']

def extract_features(p, n):
    feats = []
    for col in signal_cols:
        s = p[col][:n]
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

raw_feat_names   = [f'{c}_{s}' for c in signal_cols for s in ['mean','std','rms','peak']]
delta_feat_names = [f'Δ{c}_{s}' for c in signal_cols for s in ['mean','std','rms','peak']]
meta_feat_names  = ['DOC', 'feed', 'material']
feat_names = raw_feat_names + delta_feat_names + meta_feat_names  # 51-dim

# ── 데이터 로드 (Run1+Interp, best config) ───────────────────────────────
all_passes = []
for p in passes:
    c, r = int(p['case']), int(p['run'])
    if (c == 2 and r == 1) or (c == 12 and r == 1):
        continue
    all_passes.append({'case': c, 'run': r, 'VB': p['VB'],
                       'DOC': float(p['DOC']), 'feed': float(p['feed']),
                       'material': int(p['material']), 'p': p})

df_vb = pd.DataFrame([{'case': x['case'], 'run': x['run'], 'VB': x['VB']} for x in all_passes])
df_vb.loc[(df_vb['run'] == 1) & df_vb['VB'].isna(), 'VB'] = 0.0

def interp_case(grp):
    grp = grp.sort_values('run').copy()
    grp['VB'] = grp['VB'].interpolate(method='index', limit_area='inside')
    return grp
df_vb = df_vb.groupby('case', group_keys=False).apply(interp_case)

pass_dict = {(x['case'], x['run']): x for x in all_passes}
valid_passes = []
for _, row in df_vb.iterrows():
    if np.isnan(row['VB']):
        continue
    item = pass_dict.get((int(row['case']), int(row['run'])))
    if item is None:
        continue
    valid_passes.append({**item, 'VB': row['VB']})

base_run = {}
for item in valid_passes:
    c = item['case']
    if c not in base_run or item['run'] < base_run[c]['run']:
        base_run[c] = item

n = int(N_FULL * RATIO)
X_all, y_all = [], []
for item in valid_passes:
    c      = item['case']
    f_raw  = extract_features(item['p'], n)
    f_base = extract_features(base_run[c]['p'], n)
    f_delt = f_raw - f_base
    meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
    X_all.append(np.concatenate([f_raw, f_delt, meta]))
    y_all.append(item['VB'])

X_all = np.array(X_all)
y_all = np.array(y_all)

sc = StandardScaler()
X_scaled = sc.fit_transform(X_all)
print(f"데이터: {X_all.shape[0]} samples × {X_all.shape[1]} features")

# ── 모델 학습 ─────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

fitted = {}
for mname, model in MODELS.items():
    model.fit(X_scaled, y_all)
    fitted[mname] = model
    print(f"{mname} 학습 완료")

# ── Feature Importance 추출 ───────────────────────────────────────────────
importances = {}

coef = np.abs(fitted['Ridge'].coef_)
importances['Ridge'] = coef / coef.sum()

importances['RF'] = fitted['RF'].feature_importances_

importances['XGB'] = fitted['XGB'].feature_importances_

perm = permutation_importance(fitted['SVR'], X_scaled, y_all,
                               n_repeats=20, random_state=42, n_jobs=-1)
imp_svr = np.clip(perm.importances_mean, 0, None)
importances['SVR'] = imp_svr / imp_svr.sum() if imp_svr.sum() > 0 else imp_svr

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()
colors = {'Ridge': 'steelblue', 'RF': 'green', 'XGB': 'orange', 'SVR': 'red'}

for ax, (mname, imp) in zip(axes, importances.items()):
    top_idx   = np.argsort(imp)[::-1][:20]
    top_imp   = imp[top_idx]
    top_names = [feat_names[i] for i in top_idx]
    ax.barh(range(20), top_imp[::-1], color=colors[mname], alpha=0.8)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_names[::-1], fontsize=9)
    ax.set_xlabel('Importance (normalized)')
    ax.set_title(f'{mname} — Top 20 Feature Importance', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance per Model  (Raw + Delta + Meta, ratio=1.0)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/feature_importance.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"저장: {save_path}")

# ── 수치 테이블 출력 ─────────────────────────────────────────────────────
df_imp = pd.DataFrame(importances, index=feat_names)

print("\n=== 모델별 Feature Importance 상위 15 ===")
for mname in MODELS:
    print(f"\n[{mname}]")
    top = df_imp[mname].sort_values(ascending=False).head(15)
    for fname, val in top.items():
        print(f"  {fname:<35} {val:.4f}")

# ── 4개 모델 공통 Top-10 ──────────────────────────────────────────────────
top10_sets = {m: set(df_imp[m].sort_values(ascending=False).head(10).index) for m in MODELS}
common = set.intersection(*top10_sets.values())
print(f"\n=== 4개 모델 공통 Top-10 feature ({len(common)}개) ===")
for f in sorted(common) if common else ['(없음)']:
    print(f"  {f}")


In [ ]:
import numpy as np, pandas as pd, copy
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ── 이전 셀에서 이어받는 변수: valid_passes, base_run, feat_names, importances, MODELS, X_all, y_all ──

# case_arr 재구성
case_arr = np.array([item['case'] for item in valid_passes])
cases    = sorted(set(case_arr))

# ── 모델별 Top-5 feature 인덱스 선택 ─────────────────────────────────────
top5_idx = {mname: np.argsort(imp)[::-1][:5]
            for mname, imp in importances.items()}

print("=== 모델별 Top-5 Feature ===")
for mname, idx in top5_idx.items():
    names = [feat_names[i] for i in idx]
    print(f"  {mname:<6}: {names}")

# ── LOCO-CV (ratio=1.0 고정) ──────────────────────────────────────────────
def loco_cv_subset(X, y, case_arr, cases, model_proto):
    rmses = []
    for tc in cases:
        tr = case_arr != tc
        te = case_arr == tc
        if te.sum() == 0:
            continue
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
        m   = copy.deepcopy(model_proto)
        m.fit(Xtr, y[tr])
        pred = m.predict(Xte)
        rmses.append(np.sqrt(mean_squared_error(y[te], pred)))
    return np.mean(rmses)

# Full-feature baseline (51-dim)
sc_full = StandardScaler()
X_full_scaled = sc_full.fit_transform(X_all)

results = []
for mname, mproto in MODELS.items():
    # full feature baseline
    rmse_full = loco_cv_subset(X_full_scaled, y_all, case_arr, cases, mproto)

    # top-5 only
    idx = top5_idx[mname]
    X_top5 = X_all[:, idx]
    sc5  = StandardScaler()
    X_top5_scaled = sc5.fit_transform(X_top5)
    rmse_top5 = loco_cv_subset(X_top5_scaled, y_all, case_arr, cases, mproto)

    results.append({'model': mname,
                    'rmse_full': rmse_full,
                    'rmse_top5': rmse_top5,
                    'delta':     rmse_top5 - rmse_full,
                    'top5_features': [feat_names[i] for i in idx]})
    print(f"{mname:<6}  full={rmse_full:.4f}  top5={rmse_top5:.4f}  Δ={rmse_top5-rmse_full:+.4f}")

df_cmp = pd.DataFrame(results)

# ── 결과 출력 ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  LOCO-CV RMSE 비교  (Full 51-dim  vs  Top-5)")
print("="*60)
print(f"{'Model':<8} {'Full':>10} {'Top-5':>10} {'Δ(Top5-Full)':>14}")
print("-"*48)
for r in results:
    sign = '▲' if r['delta'] > 0 else '▼'
    print(f"{r['model']:<8} {r['rmse_full']:>10.4f} {r['rmse_top5']:>10.4f}  {sign}{abs(r['delta']):.4f}")

# ── 시각화 ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_names = [r['model'] for r in results]
rmse_full   = [r['rmse_full'] for r in results]
rmse_top5   = [r['rmse_top5'] for r in results]
colors_map  = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}
bar_colors  = [colors_map[m] for m in model_names]

x = np.arange(len(model_names))
w = 0.35
axes[0].bar(x - w/2, rmse_full, w, label='Full (51-dim)', color=bar_colors, alpha=0.5)
axes[0].bar(x + w/2, rmse_top5, w, label='Top-5',         color=bar_colors, alpha=0.95)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_ylabel('LOCO-CV RMSE (mm)'); axes[0].set_title('Full vs Top-5 Features')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# 각 모델의 top-5 feature 텍스트 표시
axes[1].axis('off')
table_data = [[r['model']] + r['top5_features'] for r in results]
col_labels = ['Model', 'Feat 1', 'Feat 2', 'Feat 3', 'Feat 4', 'Feat 5']
tbl = axes[1].table(cellText=table_data, colLabels=col_labels,
                    loc='center', cellLoc='left')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.scale(1.0, 2.0)
axes[1].set_title('Top-5 Features per Model', fontsize=12, fontweight='bold')

plt.suptitle('Retrain with Top-5 Features (ratio=1.0, LOCO-CV)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/top5_retrain.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")


In [ ]:
import numpy as np, pandas as pd, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import kurtosis as sp_kurtosis, skew as sp_skew
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

# ── 상수 (세션 변수 재사용: valid_passes, base_run) ─────────────────────
N_FULL      = 9000
FS          = 250          # Hz
RATIO       = 1.0
signal_cols = ['smcAC','smcDC','vib_table','vib_spindle','AE_table','AE_spindle']
STAT_NAMES  = ['mean','std','rms','peak','kurtosis','skewness','spec_centroid']

def extract_features_v2(p, n):
    freqs = np.fft.rfftfreq(n, d=1.0/FS)
    feats = []
    for col in signal_cols:
        s = p[col][:n].astype(float)
        fft_mag = np.abs(np.fft.rfft(s))
        denom   = fft_mag.sum() + 1e-8
        feats += [
            s.mean(),
            s.std(),
            np.sqrt((s**2).mean()),
            np.abs(s).max(),
            float(sp_kurtosis(s, fisher=True)),
            float(sp_skew(s)),
            float((freqs * fft_mag).sum() / denom),
        ]
    return np.array(feats, dtype=float)

raw_names   = [f'{c}_{s}' for c in signal_cols for s in STAT_NAMES]   # 42
delta_names = [f'Δ{c}_{s}' for c in signal_cols for s in STAT_NAMES]  # 42
meta_names  = ['DOC', 'feed', 'material']
feat_names_v2 = raw_names + delta_names + meta_names                   # 87

# ── 피처 행렬 구성 ────────────────────────────────────────────────────────
n = int(N_FULL * RATIO)
X_v2, y_v2, case_arr_v2 = [], [], []
for item in valid_passes:
    c      = item['case']
    f_raw  = extract_features_v2(item['p'], n)
    f_base = extract_features_v2(base_run[c]['p'], n)
    meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
    X_v2.append(np.concatenate([f_raw, f_raw - f_base, meta]))
    y_v2.append(item['VB'])
    case_arr_v2.append(c)

X_v2      = np.array(X_v2)
y_v2      = np.array(y_v2)
case_arr_v2 = np.array(case_arr_v2)
cases_v2  = sorted(set(case_arr_v2))
print(f"피처 행렬: {X_v2.shape}  (samples × features)")

# ── LOCO-CV ───────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

rmse_v2 = {}
for mname, mproto in MODELS.items():
    rmses = []
    for tc in cases_v2:
        tr = case_arr_v2 != tc
        te = case_arr_v2 == tc
        if te.sum() == 0:
            continue
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X_v2[tr]); Xte = sc.transform(X_v2[te])
        m   = copy.deepcopy(mproto)
        m.fit(Xtr, y_v2[tr])
        pred = m.predict(Xte)
        rmses.append(np.sqrt(mean_squared_error(y_v2[te], pred)))
    rmse_v2[mname] = np.mean(rmses)

# 이전 51-dim 기준 (셀 K3nvCvdtI2XZ 결과)
rmse_baseline = {'Ridge': 0.1474, 'RF': 0.1174, 'XGB': 0.1154, 'SVR': 0.1187}

print("\n" + "="*62)
print("  LOCO-CV RMSE 비교  (51-dim baseline  vs  87-dim +3stats)")
print("="*62)
print(f"{'Model':<8} {'Baseline(51)':>14} {'New(87)':>10} {'Δ':>10}")
print("-"*48)
for mname in MODELS:
    delta = rmse_v2[mname] - rmse_baseline[mname]
    sign  = '▼' if delta < 0 else '▲'
    print(f"{mname:<8} {rmse_baseline[mname]:>14.4f} {rmse_v2[mname]:>10.4f}  {sign}{abs(delta):.4f}")

# ── 전체 데이터로 학습 후 Feature Importance ─────────────────────────────
sc_all = StandardScaler()
X_v2_scaled = sc_all.fit_transform(X_v2)

fitted_v2 = {}
for mname, model in MODELS.items():
    model.fit(X_v2_scaled, y_v2)
    fitted_v2[mname] = model

imp_v2 = {}
coef = np.abs(fitted_v2['Ridge'].coef_)
imp_v2['Ridge'] = coef / coef.sum()
imp_v2['RF']  = fitted_v2['RF'].feature_importances_
imp_v2['XGB'] = fitted_v2['XGB'].feature_importances_
perm = permutation_importance(fitted_v2['SVR'], X_v2_scaled, y_v2,
                               n_repeats=20, random_state=42, n_jobs=-1)
imp_svr = np.clip(perm.importances_mean, 0, None)
imp_v2['SVR'] = imp_svr / imp_svr.sum() if imp_svr.sum() > 0 else imp_svr

# ── Feature Importance 시각화 ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(20, 15))
axes = axes.flatten()
colors_map = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}

for ax, (mname, imp) in zip(axes, imp_v2.items()):
    top_idx   = np.argsort(imp)[::-1][:20]
    top_imp   = imp[top_idx]
    top_names = [feat_names_v2[i] for i in top_idx]
    ax.barh(range(20), top_imp[::-1], color=colors_map[mname], alpha=0.85)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_names[::-1], fontsize=8.5)
    ax.set_xlabel('Importance (normalized)')
    ax.set_title(f'{mname} — Top 20  (RMSE: {rmse_v2[mname]:.4f})', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance — Raw+Delta+Meta (+kurtosis/skewness/spec_centroid), ratio=1.0',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/feature_importance_v2.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")

# ── 새 feature 기여도 확인 ────────────────────────────────────────────────
new_stats = ['kurtosis', 'skewness', 'spec_centroid']
print("\n=== 신규 추가 통계량 importance 합산 (모델별) ===")
df_imp_v2 = pd.DataFrame(imp_v2, index=feat_names_v2)
for mname in MODELS:
    total_new = sum(
        df_imp_v2.loc[[f for f in feat_names_v2 if any(s in f for s in new_stats)], mname].sum()
        for _ in [1]  # dummy loop for block
    )
    print(f"  {mname:<6}: {total_new:.4f}  ({total_new*100:.1f}%)")

# ── 모델별 Top-5 출력 ─────────────────────────────────────────────────────
print("\n=== 모델별 Top-5 Feature (87-dim) ===")
for mname, imp in imp_v2.items():
    top5 = [feat_names_v2[i] for i in np.argsort(imp)[::-1][:5]]
    print(f"  {mname:<6}: {top5}")


In [ ]:
import numpy as np, pandas as pd, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import kurtosis as sp_kurtosis, skew as sp_skew
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

N_FULL     = 9000
FS         = 250
RATIO      = 1.0
sig_cols   = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']   # vib 2개 제거
STAT_NAMES = ['mean','std','rms','peak','kurtosis','skewness','spec_centroid']

def extract_features_v3(p, n):
    freqs = np.fft.rfftfreq(n, d=1.0/FS)
    feats = []
    for col in sig_cols:
        s       = p[col][:n].astype(float)
        fft_mag = np.abs(np.fft.rfft(s))
        denom   = fft_mag.sum() + 1e-8
        feats += [
            s.mean(),
            s.std(),
            np.sqrt((s**2).mean()),
            np.abs(s).max(),
            float(sp_kurtosis(s, fisher=True)),
            float(sp_skew(s)),
            float((freqs * fft_mag).sum() / denom),
        ]
    return np.array(feats, dtype=float)

raw_names_v3   = [f'{c}_{s}' for c in sig_cols for s in STAT_NAMES]   # 28
delta_names_v3 = [f'Δ{c}_{s}' for c in sig_cols for s in STAT_NAMES]  # 28
feat_names_v3  = raw_names_v3 + delta_names_v3 + ['DOC','feed','material']  # 59

# ── 피처 행렬 구성 ────────────────────────────────────────────────────────
n = int(N_FULL * RATIO)
X_v3, y_v3, case_arr_v3 = [], [], []
for item in valid_passes:
    c      = item['case']
    f_raw  = extract_features_v3(item['p'], n)
    f_base = extract_features_v3(base_run[c]['p'], n)
    meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
    X_v3.append(np.concatenate([f_raw, f_raw - f_base, meta]))
    y_v3.append(item['VB'])
    case_arr_v3.append(c)

X_v3       = np.array(X_v3)
y_v3       = np.array(y_v3)
case_arr_v3 = np.array(case_arr_v3)
cases_v3   = sorted(set(case_arr_v3))
print(f"피처 행렬: {X_v3.shape}  (samples × features)")

# ── LOCO-CV ───────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

rmse_v3 = {}
for mname, mproto in MODELS.items():
    rmses = []
    for tc in cases_v3:
        tr = case_arr_v3 != tc
        te = case_arr_v3 == tc
        if te.sum() == 0:
            continue
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X_v3[tr]); Xte = sc.transform(X_v3[te])
        m   = copy.deepcopy(mproto)
        m.fit(Xtr, y_v3[tr])
        pred = m.predict(Xte)
        rmses.append(np.sqrt(mean_squared_error(y_v3[te], pred)))
    rmse_v3[mname] = np.mean(rmses)

# 이전 실험 결과 (하드코딩)
prev = {
    '51-dim\n(6sig×4stat)':  {'Ridge':0.1474,'RF':0.1174,'XGB':0.1154,'SVR':0.1187},
    '87-dim\n(6sig×7stat)':  {'Ridge':0.1622,'RF':0.1205,'XGB':0.1200,'SVR':0.1365},
    '59-dim\n(4sig×7stat)':  rmse_v3,
}

print("\n" + "="*68)
print("  LOCO-CV RMSE 비교  (6sig×4stat / 6sig×7stat / 4sig×7stat)")
print("="*68)
print(f"{'Model':<8}", end='')
for label in prev:
    print(f"  {label.replace(chr(10),' '):<18}", end='')
print()
print("-"*68)
for mname in MODELS:
    print(f"{mname:<8}", end='')
    for d in prev.values():
        print(f"  {d[mname]:>18.4f}", end='')
    print()

# ── 전체 데이터 학습 후 Feature Importance ───────────────────────────────
sc_all = StandardScaler()
X_v3_scaled = sc_all.fit_transform(X_v3)

fitted_v3, imp_v3 = {}, {}
for mname, model in MODELS.items():
    model.fit(X_v3_scaled, y_v3)
    fitted_v3[mname] = model

coef = np.abs(fitted_v3['Ridge'].coef_)
imp_v3['Ridge'] = coef / coef.sum()
imp_v3['RF']  = fitted_v3['RF'].feature_importances_
imp_v3['XGB'] = fitted_v3['XGB'].feature_importances_
perm = permutation_importance(fitted_v3['SVR'], X_v3_scaled, y_v3,
                               n_repeats=20, random_state=42, n_jobs=-1)
imp_svr = np.clip(perm.importances_mean, 0, None)
imp_v3['SVR'] = imp_svr / imp_svr.sum() if imp_svr.sum() > 0 else imp_svr

# ── 시각화 1: RMSE 비교 막대 그래프 ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
model_names = list(MODELS.keys())
colors_map  = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}
bar_colors  = [colors_map[m] for m in model_names]
x = np.arange(len(model_names))
w = 0.25
exp_labels = ['6sig×4stat\n(51-dim)', '6sig×7stat\n(87-dim)', '4sig×7stat\n(59-dim)']
offsets    = [-w, 0, w]
alphas     = [0.45, 0.65, 0.95]

for off, alpha, exp_label, d in zip(offsets, alphas, exp_labels, prev.values()):
    vals = [d[m] for m in model_names]
    axes[0].bar(x + off, vals, w, label=exp_label,
                color=bar_colors, alpha=alpha)

axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_ylabel('LOCO-CV RMSE (mm)'); axes[0].set_title('RMSE 비교 (3가지 피처 셋)')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)

# ── 시각화 2: Feature Importance (RF, 가장 안정적) ──────────────────────
ax = axes[1]
top_idx   = np.argsort(imp_v3['RF'])[::-1][:20]
top_imp   = imp_v3['RF'][top_idx]
top_names = [feat_names_v3[i] for i in top_idx]
ax.barh(range(20), top_imp[::-1], color='green', alpha=0.85)
ax.set_yticks(range(20)); ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel('Importance'); ax.set_title('RF Top-20 Feature Importance\n(4sig×7stat, 59-dim)', fontsize=11)
ax.grid(axis='x', alpha=0.3)

plt.suptitle('Experiment: Remove vib_table & vib_spindle', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/feature_importance_v3.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")

# ── 모델별 Top-5 ──────────────────────────────────────────────────────────
print("\n=== 모델별 Top-5 Feature (4sig×7stat, 59-dim) ===")
for mname, imp in imp_v3.items():
    top5 = [feat_names_v3[i] for i in np.argsort(imp)[::-1][:5]]
    print(f"  {mname:<6}: {top5}")


In [ ]:
import numpy as np, pandas as pd, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

N_FULL   = 9000
RATIO    = 1.0
sig_cols = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']

def extract_features_v4(p, n):
    feats = []
    for col in sig_cols:
        s = p[col][:n].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

raw_names_v4   = [f'{c}_{s}' for c in sig_cols for s in ['mean','std','rms','peak']]  # 16
delta_names_v4 = [f'Δ{c}_{s}' for c in sig_cols for s in ['mean','std','rms','peak']] # 16
feat_names_v4  = raw_names_v4 + delta_names_v4 + ['DOC','feed','material']             # 35

# ── 피처 행렬 구성 ────────────────────────────────────────────────────────
n = int(N_FULL * RATIO)
X_v4, y_v4, case_arr_v4 = [], [], []
for item in valid_passes:
    c      = item['case']
    f_raw  = extract_features_v4(item['p'], n)
    f_base = extract_features_v4(base_run[c]['p'], n)
    meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
    X_v4.append(np.concatenate([f_raw, f_raw - f_base, meta]))
    y_v4.append(item['VB'])
    case_arr_v4.append(c)

X_v4       = np.array(X_v4)
y_v4       = np.array(y_v4)
case_arr_v4 = np.array(case_arr_v4)
cases_v4   = sorted(set(case_arr_v4))
print(f"피처 행렬: {X_v4.shape}  (samples × features)")

# ── LOCO-CV ───────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

rmse_v4 = {}
for mname, mproto in MODELS.items():
    rmses = []
    for tc in cases_v4:
        tr = case_arr_v4 != tc
        te = case_arr_v4 == tc
        if te.sum() == 0:
            continue
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X_v4[tr]); Xte = sc.transform(X_v4[te])
        m   = copy.deepcopy(mproto)
        m.fit(Xtr, y_v4[tr])
        pred = m.predict(Xte)
        rmses.append(np.sqrt(mean_squared_error(y_v4[te], pred)))
    rmse_v4[mname] = np.mean(rmses)

# ── 비교 테이블 ───────────────────────────────────────────────────────────
baseline_51 = {'Ridge':0.1474,'RF':0.1174,'XGB':0.1154,'SVR':0.1187}

print("\n" + "="*52)
print("  LOCO-CV RMSE  (6sig×4stat  vs  4sig×4stat)")
print("="*52)
print(f"{'Model':<8} {'6sig×4stat (51)':>16} {'4sig×4stat (35)':>16} {'Δ':>8}")
print("-"*52)
for mname in MODELS:
    d = rmse_v4[mname] - baseline_51[mname]
    sign = '▼' if d < 0 else '▲'
    print(f"{mname:<8} {baseline_51[mname]:>16.4f} {rmse_v4[mname]:>16.4f}  {sign}{abs(d):.4f}")

# ── Feature Importance ────────────────────────────────────────────────────
sc_all = StandardScaler()
X_v4_scaled = sc_all.fit_transform(X_v4)

imp_v4 = {}
for mname, model in MODELS.items():
    model.fit(X_v4_scaled, y_v4)

coef = np.abs(MODELS['Ridge'].coef_)
imp_v4['Ridge'] = coef / coef.sum()
imp_v4['RF']  = MODELS['RF'].feature_importances_
imp_v4['XGB'] = MODELS['XGB'].feature_importances_
perm = permutation_importance(MODELS['SVR'], X_v4_scaled, y_v4,
                               n_repeats=20, random_state=42, n_jobs=-1)
imp_svr = np.clip(perm.importances_mean, 0, None)
imp_v4['SVR'] = imp_svr / imp_svr.sum() if imp_svr.sum() > 0 else imp_svr

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 13))
axes = axes.flatten()
colors_map = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}

for ax, (mname, imp) in zip(axes, imp_v4.items()):
    top_idx   = np.argsort(imp)[::-1]
    top_imp   = imp[top_idx]
    top_names = [feat_names_v4[i] for i in top_idx]
    n_show    = len(feat_names_v4)  # 35개 전부 표시
    ax.barh(range(n_show), top_imp[::-1], color=colors_map[mname], alpha=0.85)
    ax.set_yticks(range(n_show))
    ax.set_yticklabels(top_names[::-1], fontsize=8)
    ax.set_xlabel('Importance (normalized)')
    rmse_str = f"RMSE: {rmse_v4[mname]:.4f}"
    ax.set_title(f'{mname} — All Features  ({rmse_str})', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Feature Importance — 4sig × 4stat + Meta  (35-dim, ratio=1.0)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/feature_importance_v4.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")

print("\n=== 모델별 Top-5 Feature ===")
for mname, imp in imp_v4.items():
    top5 = [feat_names_v4[i] for i in np.argsort(imp)[::-1][:5]]
    print(f"  {mname:<6}: {top5}")


In [ ]:
import numpy as np, pandas as pd, copy, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

N_FULL   = 9000
RATIOS   = [r / 10 for r in range(1, 11)]
sig_cols = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']

def extract_features_v4(p, n):
    feats = []
    for col in sig_cols:
        s = p[col][:n].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

results = []
for ratio in RATIOS:
    n = int(N_FULL * ratio)
    X, y, case_arr = [], [], []
    for item in valid_passes:
        c      = item['case']
        f_raw  = extract_features_v4(item['p'], n)
        f_base = extract_features_v4(base_run[c]['p'], n)
        meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
        X.append(np.concatenate([f_raw, f_raw - f_base, meta]))
        y.append(item['VB'])
        case_arr.append(c)

    X        = np.array(X)
    y        = np.array(y)
    case_arr = np.array(case_arr)
    cases    = sorted(set(case_arr))

    for mname, mproto in MODELS.items():
        rmses = []
        for tc in cases:
            tr = case_arr != tc
            te = case_arr == tc
            if te.sum() == 0:
                continue
            sc  = StandardScaler()
            Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
            m   = copy.deepcopy(mproto)
            m.fit(Xtr, y[tr])
            pred = m.predict(Xte)
            rmses.append(np.sqrt(mean_squared_error(y[te], pred)))
        results.append({'ratio': ratio, 'model': mname, 'rmse': np.mean(rmses)})

    print(f"  ratio={ratio:.1f}  done")

df_res = pd.DataFrame(results)

# ── 결과 테이블 ───────────────────────────────────────────────────────────
print("\n" + "="*52)
print("  LOCO-CV RMSE  [4sig×4stat, 35-dim]")
print("="*52)
print(df_res.pivot(index='ratio', columns='model', values='rmse').round(4).to_string())

# 모델별 최적 ratio
print("\n=== 모델별 최적 ratio ===")
for mname in MODELS:
    sub  = df_res[df_res['model'] == mname]
    best = sub.loc[sub['rmse'].idxmin()]
    print(f"  {mname:<6}  ratio={best['ratio']:.1f}  RMSE={best['rmse']:.4f}")

# ── 6sig×4stat 기존 결과 (cell 18 하드코딩) ──────────────────────────────
baseline_by_ratio = {
    'Ridge': {0.1:0.2119,0.2:0.1775,0.3:0.1644,0.4:0.1613,0.5:0.1591,
              0.6:0.1580,0.7:0.1554,0.8:0.1530,0.9:0.1496,1.0:0.1474},
    'RF':    {0.1:0.1875,0.2:0.1481,0.3:0.1340,0.4:0.1280,0.5:0.1233,
              0.6:0.1225,0.7:0.1209,0.8:0.1196,0.9:0.1181,1.0:0.1174},
    'XGB':   {0.1:0.1892,0.2:0.1507,0.3:0.1352,0.4:0.1276,0.5:0.1221,
              0.6:0.1211,0.7:0.1193,0.8:0.1175,0.9:0.1162,1.0:0.1154},
    'SVR':   {0.1:0.2027,0.2:0.1654,0.3:0.1516,0.4:0.1438,0.5:0.1376,
              0.6:0.1341,0.7:0.1294,0.8:0.1268,0.9:0.1228,1.0:0.1187},
}

# ── 시각화 ────────────────────────────────────────────────────────────────
colors = {'Ridge':'steelblue','RF':'green','XGB':'orange','SVR':'red'}
x_pct  = [r * 100 for r in RATIOS]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, mname in zip(axes, MODELS):
    sub_new = df_res[df_res['model'] == mname].sort_values('ratio')
    base_vals = [baseline_by_ratio[mname][r] for r in RATIOS]

    ax.plot(x_pct, base_vals,           color=colors[mname], linestyle='--',
            linewidth=1.8, marker='s', markersize=5, alpha=0.55, label='6sig×4stat (51-dim)')
    ax.plot(x_pct, sub_new['rmse'].values, color=colors[mname], linestyle='-',
            linewidth=2.0, marker='o', markersize=5, label='4sig×4stat (35-dim)')

    best_new = sub_new.loc[sub_new['rmse'].idxmin()]
    ax.axvline(best_new['ratio']*100, color=colors[mname], linestyle=':',
               linewidth=1.2, alpha=0.7)
    ax.axvline(25, color='gray', linestyle='--', linewidth=1, alpha=0.4, label='entry end (25%)')

    ax.set_title(f'{mname}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Input length (%)'); ax.set_ylabel('RMSE (mm)')
    ax.set_xticks(x_pct); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.annotate(f"best={best_new['rmse']:.4f}\n@{int(best_new['ratio']*100)}%",
                xy=(best_new['ratio']*100, best_new['rmse']),
                xytext=(8, 8), textcoords='offset points', fontsize=8,
                arrowprops=dict(arrowstyle='->', color='black', lw=1))

plt.suptitle('RMSE vs Input Ratio — 4sig×4stat (35-dim)  vs  6sig×4stat (51-dim)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/ratio_rmse_v4.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")


In [ ]:
import torch, torch.nn as nn
import numpy as np, pandas as pd, copy, warnings, time
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

N_FULL  = 9000
SIG     = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
IN_DIM  = len(SIG) * 4   # 16
HIDDEN  = 32
LAYERS  = 2
EPOCHS  = 200
LR      = 1e-3

def extract_raw(p, n=N_FULL):
    feats = []
    for col in SIG:
        s = p[col][:n].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── 케이스별 run 시퀀스 구성 ─────────────────────────────────────────────
case_items = defaultdict(list)
for item in valid_passes:
    case_items[item['case']].append(item)

case_seqs = {}
for c, items in case_items.items():
    items = sorted(items, key=lambda x: x['run'])
    X = np.array([extract_raw(it['p']) for it in items])   # (T, 16)
    y = np.array([it['VB'] for it in items], dtype=float)
    case_seqs[c] = (X, y)

print("케이스별 시퀀스 길이 및 VB 범위:")
for c in CASES:
    X, y = case_seqs[c]
    v = y[~np.isnan(y)]
    print(f"  C{c:02d}: {X.shape[0]:2d} runs  VB [{v.min():.3f} ~ {v.max():.3f}]")

# ── 모델 정의 ─────────────────────────────────────────────────────────────
class SeqModel(nn.Module):
    def __init__(self, in_dim, hidden=32, n_layers=2, kind='LSTM'):
        super().__init__()
        rnn_cls = {'RNN': nn.RNN, 'GRU': nn.GRU, 'LSTM': nn.LSTM}[kind]
        kw = dict(input_size=in_dim, hidden_size=hidden,
                  num_layers=n_layers, batch_first=True)
        if n_layers > 1:
            kw['dropout'] = 0.1
        if kind == 'RNN':
            kw['nonlinearity'] = 'tanh'
        self.rnn  = rnn_cls(**kw)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):      # x: (B, T, D)
        out, _ = self.rnn(x)   # (B, T, H)
        return self.head(out).squeeze(-1)  # (B, T)

# ── 배치 생성 & 학습 ─────────────────────────────────────────────────────
def make_batch(train_cases, scaled_seqs):
    xs, ys = [], []
    for c in train_cases:
        X_s, y = scaled_seqs[c]
        xs.append(torch.FloatTensor(X_s))
        ys.append(torch.FloatTensor(y))
    xs_pad = pad_sequence(xs, batch_first=True).to(DEVICE)
    ys_pad = pad_sequence(ys, batch_first=True, padding_value=float('nan')).to(DEVICE)
    return xs_pad, ys_pad

def train_model(kind, train_cases, scaled_seqs):
    model = SeqModel(IN_DIM, hidden=HIDDEN, n_layers=LAYERS, kind=kind).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sch   = CosineAnnealingLR(opt, T_max=EPOCHS)
    xs_pad, ys_pad = make_batch(train_cases, scaled_seqs)
    for _ in range(EPOCHS):
        model.train()
        opt.zero_grad()
        pred = model(xs_pad)
        mask = ~torch.isnan(ys_pad)
        loss = ((pred[mask] - ys_pad[mask])**2).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
    return model

# ── LOCO-CV ───────────────────────────────────────────────────────────────
seq_results = {}
all_preds   = {}

for kind in ['RNN', 'GRU', 'LSTM']:
    print(f"\n[{kind}] LOCO-CV 진행 중...")
    t0 = time.time()
    rmses, preds_by_case = [], {}

    for test_c in CASES:
        train_cs = [c for c in CASES if c != test_c]
        scaler = StandardScaler().fit(np.vstack([case_seqs[c][0] for c in train_cs]))
        scaled_seqs = {c: (scaler.transform(case_seqs[c][0]), case_seqs[c][1]) for c in CASES}

        model = train_model(kind, train_cs, scaled_seqs)

        model.eval()
        X_te = scaled_seqs[test_c][0]
        y_te = case_seqs[test_c][1]
        with torch.no_grad():
            x_t = torch.FloatTensor(X_te).unsqueeze(0).to(DEVICE)
            pred = model(x_t).cpu().numpy().squeeze(0)

        mask = ~np.isnan(y_te)
        rmses.append(np.sqrt(mean_squared_error(y_te[mask], pred[mask])))
        preds_by_case[test_c] = (y_te, pred)

    seq_results[kind] = {'rmses': rmses, 'mean': float(np.mean(rmses))}
    all_preds[kind]   = preds_by_case
    print(f"  {kind}: 평균 RMSE = {np.mean(rmses):.4f}  ({time.time()-t0:.1f}s)")

# ── 결과 테이블 ───────────────────────────────────────────────────────────
baseline    = {'Ridge': 0.1509, 'RF': 0.1181, 'XGB': 0.1153, 'SVR': 0.1167}
best_static = min(baseline.values())

print("\n" + "="*52)
print("  LOCO-CV RMSE — Static vs Sequence Models")
print("="*52)
for m, r in baseline.items():
    print(f"  [Static] {m:<5} {r:>8.4f}")
print()
for kind in ['RNN', 'GRU', 'LSTM']:
    r    = seq_results[kind]['mean']
    diff = r - best_static
    sign = '▼' if diff < 0 else '▲'
    print(f"  [Seq]    {kind:<5} {r:>8.4f}  {sign}{abs(diff):.4f}")

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

model_names = list(baseline.keys()) + ['RNN', 'GRU', 'LSTM']
rmse_vals   = list(baseline.values()) + [seq_results[k]['mean'] for k in ['RNN','GRU','LSTM']]
bar_colors  = ['#b0c4de']*4 + ['#ff9999', '#90ee90', '#ffd580']
bars = axes[0].bar(model_names, rmse_vals, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[0].axhline(best_static, color='steelblue', linestyle='--', linewidth=1.2,
                label=f'best static ({best_static:.4f})')
axes[0].set_ylabel('LOCO-CV RMSE (mm)')
axes[0].set_title('Static vs Sequence Models')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8.5)

ax = axes[1]
c0 = CASES[0]
y_true = all_preds['LSTM'][c0][0]
runs   = np.arange(1, len(y_true) + 1)
mask   = ~np.isnan(y_true)
ax.plot(runs[mask], y_true[mask], 'ko-', markersize=5, linewidth=1.8, label='True VB', zorder=5)
styles = {'RNN': ('r', ':'), 'GRU': ('g', '-.'), 'LSTM': ('b', '--')}
for kind, (color, ls) in styles.items():
    yp   = all_preds[kind][c0][1]
    rmse = np.sqrt(mean_squared_error(y_true[mask], yp[mask]))
    ax.plot(runs, yp, color=color, linestyle=ls, linewidth=1.8, label=f'{kind} (RMSE={rmse:.4f})')
ax.set_xlabel('Run'); ax.set_ylabel('VB (mm)')
ax.set_title(f'VB 예측 궤적 — Case {c0:02d}')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(f'Sequence Models (run-to-run, 4sig×4stat raw, hidden={HIDDEN}, layers={LAYERS})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/seq_model_results.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")

print("\n=== 케이스별 RMSE ===")
print(f"{'Case':<8}", end='')
for kind in ['RNN', 'GRU', 'LSTM']:
    print(f"  {kind:>8}", end='')
print()
for i, c in enumerate(CASES):
    print(f"  C{c:02d}   ", end='')
    for kind in ['RNN', 'GRU', 'LSTM']:
        print(f"  {seq_results[kind]['rmses'][i]:>8.4f}", end='')
    print()


In [ ]:
# ── 의존성: SETUP CELL (셀 0) 실행 후 독립 실행 가능 ─────────────────────
import torch, torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import numpy as np, warnings, time
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스: {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type == 'cuda' else ''))

# case_seqs_delta, CASES: Setup 셀에서 정의됨
IN_DIM = 16   # 4sig × 4stat delta
HIDDEN = 32
LAYERS = 2
EPOCHS = 200  # 배치 학습으로 수렴 빠름 → 400→200
LR     = 1e-3

# ── 모델 ─────────────────────────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, in_dim, hidden=32, n_layers=2):
        super().__init__()
        self.gru  = nn.GRU(in_dim, hidden, n_layers, batch_first=True,
                           dropout=0.1 if n_layers > 1 else 0)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: (B, T, D) → (B, T)
        out, _ = self.gru(x)
        return self.head(out).squeeze(-1)

# ── 배치 학습: 케이스별 시퀀스를 패딩해 한 번에 forward ──────────────────
def make_batch(train_cases, scaled_seqs):
    """스케일된 시퀀스들을 패딩 배치로 변환. scaler.transform 없음."""
    xs, ys, lengths = [], [], []
    for c in train_cases:
        X_s, y = scaled_seqs[c]
        xs.append(torch.FloatTensor(X_s))
        ys.append(torch.FloatTensor(y))
        lengths.append(X_s.shape[0])
    xs_pad = pad_sequence(xs, batch_first=True)                            # (B, T_max, D)
    ys_pad = pad_sequence(ys, batch_first=True, padding_value=float('nan')) # (B, T_max)
    return xs_pad.to(DEVICE), ys_pad.to(DEVICE), torch.tensor(lengths)

def train_gru(train_cases, scaled_seqs):
    model = GRUModel(IN_DIM, HIDDEN, LAYERS).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    # 배치는 fold 안에서 고정이므로 epoch 밖에서 한 번만 생성
    xs_pad, ys_pad, lengths = make_batch(train_cases, scaled_seqs)

    for _ in range(EPOCHS):
        model.train()
        opt.zero_grad()
        pred = model(xs_pad)               # (B, T_max)
        mask = ~torch.isnan(ys_pad)
        loss = ((pred[mask] - ys_pad[mask]) ** 2).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sch.step()
    return model

# ── LOCO-CV ───────────────────────────────────────────────────────────────
print(f"\n[GRU-CumDelta] LOCO-CV  ({len(CASES)} folds × {EPOCHS} epochs, 배치 학습)...")
t0 = time.time()
rmses, preds_by_case = [], {}

for test_c in CASES:
    train_cs = [c for c in CASES if c != test_c]

    # scaler: fold 시작 시 한 번만 fit, 이후 모든 시퀀스에 적용
    scaler = StandardScaler().fit(np.vstack([case_seqs_delta[c][0] for c in train_cs]))
    scaled_seqs = {c: (scaler.transform(case_seqs_delta[c][0]), case_seqs_delta[c][1])
                   for c in CASES}

    model = train_gru(train_cs, scaled_seqs)

    model.eval()
    X_te_s, y_te = scaled_seqs[test_c]
    with torch.no_grad():
        pred = model(
            torch.FloatTensor(X_te_s).unsqueeze(0).to(DEVICE)
        ).squeeze(0).cpu().numpy()

    mask = ~np.isnan(y_te)
    rmses.append(np.sqrt(mean_squared_error(y_te[mask], pred[mask])))
    preds_by_case[test_c] = (y_te, pred)

elapsed = time.time() - t0
mean_rmse = float(np.mean(rmses))
print(f"  완료: {elapsed:.1f}초  ({elapsed/len(CASES):.1f}초/fold)")
print(f"  GRU-CumDelta 평균 RMSE = {mean_rmse:.4f}")

# ── 결과 비교 ─────────────────────────────────────────────────────────────
BASELINE = {'Ridge': 0.1509, 'RF': 0.1181, 'XGB': 0.1153, 'SVR': 0.1167}

print("\n" + "="*48)
print("  LOCO-CV RMSE 비교")
print("="*48)
for name, r in BASELINE.items():
    print(f"  [Static] {name:<6} {r:.4f}")
diff = mean_rmse - min(BASELINE.values())
sign = '▼' if diff < 0 else '▲'
print(f"  [Seq]    GRU-CumDelta  {mean_rmse:.4f}  {sign}{abs(diff):.4f} vs best static")

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_names  = list(BASELINE.keys()) + ['GRU-CumDelta']
all_vals   = list(BASELINE.values()) + [mean_rmse]
bar_colors = ['#b0c4de']*4 + ['#4caf50']
bars = axes[0].bar(all_names, all_vals, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[0].axhline(min(BASELINE.values()), color='steelblue', linestyle='--',
                linewidth=1.2, label=f'best static ({min(BASELINE.values()):.4f})')
axes[0].set_ylabel('LOCO-CV RMSE (mm)')
axes[0].set_title('Static vs GRU-CumDelta')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, all_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

c0 = CASES[0]
y_true, y_pred = preds_by_case[c0]
runs = np.arange(1, len(y_true) + 1)
mask = ~np.isnan(y_true)
r0   = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
axes[1].plot(runs[mask], y_true[mask], 'ko-', markersize=5, lw=1.8, label='True VB')
axes[1].plot(runs, y_pred, 'g--', lw=1.8, label=f'GRU-CumDelta (RMSE={r0:.4f})')
axes[1].set_xlabel('Run'); axes[1].set_ylabel('VB (mm)')
axes[1].set_title(f'VB 예측 궤적 — Case {c0:02d}')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle(f'GRU-CumDelta  (device={DEVICE.type.upper()}, hidden={HIDDEN}, '
             f'layers={LAYERS}, epochs={EPOCHS}, batched)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/gru_cumdelta_results.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")

print("\n=== 케이스별 RMSE ===")
for i, c in enumerate(CASES):
    print(f"  C{c:02d}: {rmses[i]:.4f}")


디바이스: cuda  (Tesla T4)

[GRU-CumDelta] LOCO-CV  (16 folds × 200 epochs, 배치 학습)...
  완료: 13.3초  (0.8초/fold)
  GRU-CumDelta 평균 RMSE = 0.1138

  LOCO-CV RMSE 비교
  [Static] Ridge  0.1509
  [Static] RF     0.1181
  [Static] XGB    0.1153
  [Static] SVR    0.1167
  [Seq]    GRU-CumDelta  0.1138  ▼0.0015 vs best static

저장: /content/drive/MyDrive/dataset/nasa_milling/gru_cumdelta_results.png

=== 케이스별 RMSE ===
  C01: 0.1893
  C02: 0.0561
  C03: 0.0517
  C04: 0.1215
  C05: 0.2056
  C06: 0.0499
  C07: 0.0887
  C08: 0.1757
  C09: 0.0808
  C10: 0.1260
  C11: 0.1089
  C12: 0.1686
  C13: 0.2073
  C14: 0.0586
  C15: 0.0405
  C16: 0.0909


In [ ]:
import torch, torch.nn as nn
import numpy as np, warnings, time
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

N_FULL  = 9000
SIG     = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
IN_DIM  = len(SIG) * 4 + 3   # 16 delta + 3 meta = 19
HIDDEN  = 32
LAYERS  = 2
EPOCHS  = 400
LR      = 1e-3

def extract_raw(p, n=N_FULL):
    feats = []
    for col in SIG:
        s = p[col][:n].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── Delta + Meta 시퀀스 구성 ─────────────────────────────────────────────
# x_t = [Δ_t (16-dim), DOC, feed, material]
# run1: Δ=0이지만 meta가 케이스 조건을 구별
case_items = defaultdict(list)
for item in valid_passes:
    case_items[item['case']].append(item)

case_seqs_dm = {}
for c, items in case_items.items():
    items  = sorted(items, key=lambda x: x['run'])
    X_raw  = np.array([extract_raw(it['p']) for it in items])
    X_delt = X_raw - X_raw[0]                                    # (T, 16) Δ
    meta   = np.array([[it['DOC'], it['feed'], it['material']]
                       for it in items], dtype=float)
    X_dm   = np.concatenate([X_delt, meta], axis=1)              # (T, 19)
    y      = np.array([it['VB'] for it in items], dtype=float)
    case_seqs_dm[c] = (X_dm, y)

print(f"입력 차원: {IN_DIM}  (16 delta + 3 meta)")
print(f"run1 입력 (C01): delta={case_seqs_dm[1][0][0,:4].round(4)}... meta={case_seqs_dm[1][0][0,16:]}")
print(f"run1 입력 (C02): delta={case_seqs_dm[2][0][0,:4].round(4)}... meta={case_seqs_dm[2][0][0,16:]}")

# ── GRU 모델 ─────────────────────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, in_dim, hidden=32, n_layers=2):
        super().__init__()
        self.gru  = nn.GRU(in_dim, hidden, n_layers, batch_first=True,
                           dropout=0.1 if n_layers > 1 else 0)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):   # x: (B, T, D)
        out, _ = self.gru(x)
        return self.head(out).squeeze(-1)  # (B, T)

# ── 배치 생성 & 학습 ─────────────────────────────────────────────────────
def make_batch(train_cases, scaled_seqs):
    xs, ys = [], []
    for c in train_cases:
        X_s, y = scaled_seqs[c]
        xs.append(torch.FloatTensor(X_s))
        ys.append(torch.FloatTensor(y))
    xs_pad = pad_sequence(xs, batch_first=True).to(DEVICE)
    ys_pad = pad_sequence(ys, batch_first=True, padding_value=float('nan')).to(DEVICE)
    return xs_pad, ys_pad

def train_gru(train_cases, scaled_seqs):
    model = GRUModel(IN_DIM, HIDDEN, LAYERS).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sch   = CosineAnnealingLR(opt, T_max=EPOCHS)
    xs_pad, ys_pad = make_batch(train_cases, scaled_seqs)
    for _ in range(EPOCHS):
        model.train()
        opt.zero_grad()
        pred = model(xs_pad)
        mask = ~torch.isnan(ys_pad)
        loss = ((pred[mask] - ys_pad[mask])**2).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
    return model

# ── LOCO-CV ───────────────────────────────────────────────────────────────
print("\n[GRU-DeltaMeta] LOCO-CV 진행 중...")
t0 = time.time()
rmses_dm, preds_dm = [], {}

for test_c in CASES:
    train_cs = [c for c in CASES if c != test_c]
    scaler   = StandardScaler().fit(np.vstack([case_seqs_dm[c][0] for c in train_cs]))
    scaled_seqs = {c: (scaler.transform(case_seqs_dm[c][0]), case_seqs_dm[c][1]) for c in CASES}

    model = train_gru(train_cs, scaled_seqs)

    model.eval()
    X_te = scaled_seqs[test_c][0]
    y_te = case_seqs_dm[test_c][1]
    with torch.no_grad():
        x_t = torch.FloatTensor(X_te).unsqueeze(0).to(DEVICE)
        pred = model(x_t).cpu().numpy().squeeze(0)

    mask = ~np.isnan(y_te)
    rmses_dm.append(np.sqrt(mean_squared_error(y_te[mask], pred[mask])))
    preds_dm[test_c] = (y_te, pred)

mean_dm = float(np.mean(rmses_dm))
print(f"  GRU-DeltaMeta RMSE = {mean_dm:.4f}  ({time.time()-t0:.1f}s)")

# ── 결과 비교 ─────────────────────────────────────────────────────────────
compare = {
    'XGB\n(Static)':        0.1153,
    'GRU-CumDelta\n(Seq)':  0.1138,   # 별도 셀 결과
    'GRU-DeltaMeta\n(Seq)': mean_dm,
}

print("\n" + "="*50)
print("  RMSE 비교")
print("="*50)
for name, r in compare.items():
    label = name.replace('\n', '-')
    diff  = r - 0.1153
    sign  = '▼' if diff < 0 else ('▲' if diff > 0 else ' ')
    print(f"  {label:<24} {r:.4f}  {sign}{abs(diff):.4f}")

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

names  = list(compare.keys())
vals   = list(compare.values())
colors = ['#b0c4de', '#4caf50', '#2e7d32']
bars   = axes[0].bar(names, vals, color=colors, edgecolor='black', linewidth=0.7)
axes[0].axhline(0.1153, color='steelblue', linestyle='--', linewidth=1.2, label='XGB best (0.1153)')
axes[0].set_ylabel('LOCO-CV RMSE (mm)')
axes[0].set_title('모델별 RMSE 비교')
axes[0].legend(fontsize=9); axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

ax = axes[1]
c0 = CASES[0]
y_true, y_pred = preds_dm[c0]
runs = np.arange(1, len(y_true) + 1)
mask = ~np.isnan(y_true)
rmse_c0 = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
ax.plot(runs[mask], y_true[mask], 'ko-', markersize=5, linewidth=1.8, label='True VB')
ax.plot(runs, y_pred, 'g--', linewidth=1.8, label=f'GRU-DeltaMeta (RMSE={rmse_c0:.4f})')
ax.set_xlabel('Run'); ax.set_ylabel('VB (mm)')
ax.set_title(f'VB 예측 궤적 — Case {c0:02d}')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(f'GRU-DeltaMeta (16 delta + 3 meta = {IN_DIM}-dim, hidden={HIDDEN}, layers={LAYERS})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/gru_deltameta_results.png'
plt.savefig(save_path, dpi=150)
plt.show()
print(f"\n저장: {save_path}")

print("\n=== 케이스별 RMSE ===")
print(f"  {'Case':<6} {'GRU-DeltaMeta':>14}")
print("  " + "-"*24)
for i, c in enumerate(CASES):
    print(f"  C{c:02d}   {rmses_dm[i]:>14.4f}")


Device: cuda
입력 차원: 19  (16 delta + 3 meta)
run1 입력 (C01): delta=[0. 0. 0. 0.]... meta=[1.5 0.5 1. ]
run1 입력 (C02): delta=[0. 0. 0. 0.]... meta=[0.75 0.5  1.  ]

[GRU-DeltaMeta] LOCO-CV 진행 중...
  GRU-DeltaMeta RMSE = 0.1027  (27.7s)

  RMSE 비교
  XGB-(Static)             0.1153   0.0000
  GRU-CumDelta-(Seq)       0.1138  ▼0.0015
  GRU-DeltaMeta-(Seq)      0.1027  ▼0.0126

저장: /content/drive/MyDrive/dataset/nasa_milling/gru_deltameta_results.png

=== 케이스별 RMSE ===
  Case    GRU-DeltaMeta
  ------------------------
  C01           0.0405
  C02           0.0528
  C03           0.0433
  C04           0.1720
  C05           0.2185
  C06           0.0421
  C07           0.1059
  C08           0.1809
  C09           0.0551
  C10           0.0775
  C11           0.0467
  C12           0.2102
  C13           0.1558
  C14           0.0975
  C15           0.0645
  C16           0.0794


In [ ]:
import torch, torch.nn as nn
import numpy as np, warnings, time, itertools
from collections import defaultdict
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

N_FULL = 9000
SIG    = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
IN_DIM = 19   # 16 delta + 3 meta

def extract_raw(p, n):
    feats = []
    for col in SIG:
        s = p[col][:n].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

def build_seqs_dm(ratio=1.0):
    n = int(N_FULL * ratio)
    ci = defaultdict(list)
    for item in valid_passes:
        ci[item['case']].append(item)
    seqs = {}
    for c, items in ci.items():
        items  = sorted(items, key=lambda x: x['run'])
        X_raw  = np.array([extract_raw(it['p'], n) for it in items])
        X_delt = X_raw - X_raw[0]
        meta   = np.array([[it['DOC'], it['feed'], it['material']] for it in items], dtype=float)
        seqs[c] = (np.concatenate([X_delt, meta], axis=1),
                   np.array([it['VB'] for it in items], dtype=float))
    return seqs

# ── 모델 ─────────────────────────────────────────────────────────────────
class GRUDeltaMeta(nn.Module):
    def __init__(self, hidden, n_layers):
        super().__init__()
        self.gru  = nn.GRU(IN_DIM, hidden, n_layers, batch_first=True,
                           dropout=0.1 if n_layers > 1 else 0)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.gru(x)[0]).squeeze(-1)

def make_batch(train_cs, scaled):
    xs = [torch.FloatTensor(scaled[c][0]) for c in train_cs]
    ys = [torch.FloatTensor(scaled[c][1]) for c in train_cs]
    return (pad_sequence(xs, batch_first=True).to(DEVICE),
            pad_sequence(ys, batch_first=True, padding_value=float('nan')).to(DEVICE))

def loco_cv(seqs, hidden, n_layers, epochs, lr):
    rmses = []
    for test_c in CASES:
        train_cs = [c for c in CASES if c != test_c]
        scaler   = StandardScaler().fit(np.vstack([seqs[c][0] for c in train_cs]))
        scaled   = {c: (scaler.transform(seqs[c][0]), seqs[c][1]) for c in CASES}

        model  = GRUDeltaMeta(hidden, n_layers).to(DEVICE)
        opt    = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        sch    = CosineAnnealingLR(opt, T_max=epochs)
        xb, yb = make_batch(train_cs, scaled)

        for _ in range(epochs):
            model.train(); opt.zero_grad()
            mask = ~torch.isnan(yb)
            loss = ((model(xb)[mask] - yb[mask])**2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()

        model.eval()
        with torch.no_grad():
            xt   = torch.FloatTensor(scaled[test_c][0]).unsqueeze(0).to(DEVICE)
            pred = model(xt).cpu().numpy().squeeze(0)
        y_te = seqs[test_c][1]
        mask = ~np.isnan(y_te)
        rmses.append(np.sqrt(mean_squared_error(y_te[mask], pred[mask])))
    return float(np.mean(rmses))

# ═══════════════════════════════════════════════════════════════
# 1단계: Hyperparameter Search (ratio=1.0 고정)
# ═══════════════════════════════════════════════════════════════
SEARCH = {
    'hidden': [16, 32, 64, 128, 256],
    'layers': [1, 2, 3, 4, 5],
    'epochs': [100, 200, 300, 400, 500],
    'lr':     [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
}
all_combos = list(itertools.product(*SEARCH.values()))
rng        = np.random.RandomState(42)
trials     = [all_combos[i] for i in rng.choice(len(all_combos), 40, replace=False)]

seqs_full = build_seqs_dm(ratio=1.0)

print("=" * 62)
print("  [1단계] Hyperparameter Search (GRU-DeltaMeta, ratio=1.0)")
print(f"  탐색 공간 {len(all_combos)}개 → 40개 랜덤 샘플")
print("=" * 62)
print(f"{'#':>3} {'HIDDEN':>7} {'LAYERS':>7} {'EPOCHS':>7} {'LR':>8}  {'RMSE':>7}  time")
print("-" * 58)

hp_results = []
t_total    = time.time()
for i, (h, l, e, lr) in enumerate(trials):
    t0   = time.time()
    rmse = loco_cv(seqs_full, h, l, e, lr)
    hp_results.append({'hidden': h, 'layers': l, 'epochs': e, 'lr': lr, 'rmse': rmse})
    print(f"{i+1:>3}  {h:>7}  {l:>7}  {e:>7}  {lr:>8.0e}  {rmse:.4f}  ({time.time()-t0:.1f}s)")

print(f"\n총 소요: {time.time()-t_total:.1f}s")

df   = pd.DataFrame(hp_results).sort_values('rmse').reset_index(drop=True)
best = df.iloc[0]

print("\n=== Top-10 ===")
print(f"{'#':>3} {'HIDDEN':>7} {'LAYERS':>7} {'EPOCHS':>7} {'LR':>8}  RMSE")
print("-" * 46)
for i, row in df.head(10).iterrows():
    print(f"{i+1:>3}  {int(row.hidden):>7}  {int(row.layers):>7}  {int(row.epochs):>7}  {row.lr:>8.0e}  {row.rmse:.4f}")

BEST_H, BEST_L, BEST_E, BEST_LR = int(best.hidden), int(best.layers), int(best.epochs), float(best.lr)
print(f"\n최적: HIDDEN={BEST_H}, LAYERS={BEST_L}, EPOCHS={BEST_E}, LR={BEST_LR:.0e} → RMSE={best.rmse:.4f}")

# ── 파라미터별 marginal RMSE 시각화 ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (col, cands, label) in zip(axes, [
    ('hidden', SEARCH['hidden'], 'HIDDEN'),
    ('layers', SEARCH['layers'], 'LAYERS'),
    ('epochs', SEARCH['epochs'], 'EPOCHS'),
    ('lr',     SEARCH['lr'],     'LR'),
]):
    means = [df[df[col] == v]['rmse'].mean() if (df[col] == v).any() else float('nan') for v in cands]
    xs    = [str(v) if col != 'lr' else f'{v:.0e}' for v in cands]
    ax.bar(xs, means, color='steelblue', alpha=0.8, edgecolor='black')
    ax.axhline(best.rmse, color='red', linestyle='--', linewidth=1.2, label=f'best={best.rmse:.4f}')
    ax.set_title(label); ax.set_ylabel('Mean RMSE'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    for j, v in enumerate(means):
        if not np.isnan(v):
            ax.text(j, v + 0.0005, f'{v:.4f}', ha='center', va='bottom', fontsize=7.5)

plt.suptitle('GRU-DeltaMeta: Hyperparameter Marginal Effect (40 random trials)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/gru_hparam_search.png', dpi=150)
plt.show()
print("저장: gru_hparam_search.png")


Device: cuda
  [1단계] Hyperparameter Search (GRU-DeltaMeta, ratio=1.0)
  탐색 공간 625개 → 40개 랜덤 샘플
  #  HIDDEN  LAYERS  EPOCHS       LR     RMSE  time
----------------------------------------------------------
  1      128        3      500     1e-03  0.1090  (35.2s)
  2      128        5      300     1e-04  0.1087  (25.1s)
  3       32        4      400     1e-04  0.1278  (29.7s)
  4       32        4      300     1e-03  0.1063  (22.4s)
  5      128        5      200     1e-04  0.1025  (17.4s)
  6      256        1      400     1e-04  0.0955  (24.4s)
  7       32        2      300     5e-03  0.1093  (17.0s)
  8       16        4      100     1e-03  0.1348  (7.9s)
  9      256        2      100     1e-03  0.0921  (8.0s)
 10       32        1      500     1e-04  0.1215  (25.1s)
 11       64        1      400     1e-04  0.1234  (20.3s)
 12       16        2      200     5e-04  0.1185  (11.1s)
 13       16        3      200     1e-04  0.2389  (12.5s)
 14       16        4      400     1e-04  

In [ ]:

# ═══════════════════════════════════════════════════════════════
# 2단계: 최적 파라미터로 입력 신호 길이 비율 실험
# ═══════════════════════════════════════════════════════════════
RATIOS = [r / 10 for r in range(1, 11)]

print("\n" + "=" * 62)
print(f"  [2단계] 입력 길이 비율 실험")
print(f"  (HIDDEN={BEST_H}, LAYERS={BEST_L}, EPOCHS={BEST_E}, LR={BEST_LR:.0e})")
print("=" * 62)

ratio_results = []
for ratio in RATIOS:
    t0   = time.time()
    seqs = build_seqs_dm(ratio=ratio)
    rmse = loco_cv(seqs, BEST_H, BEST_L, BEST_E, BEST_LR)
    ratio_results.append({'ratio': ratio, 'rmse': rmse})
    print(f"  ratio={ratio:.1f}  RMSE={rmse:.4f}  ({time.time()-t0:.1f}s)")

df_ratio  = pd.DataFrame(ratio_results)
best_row  = df_ratio.loc[df_ratio['rmse'].idxmin()]
print(f"\n최적 ratio: {best_row.ratio:.1f}  RMSE={best_row.rmse:.4f}")

# XGB baseline (4sig×4stat, 35-dim — 셀 29 결과)
xgb_ratio = {0.1:0.1892, 0.2:0.1507, 0.3:0.1352, 0.4:0.1276, 0.5:0.1221,
             0.6:0.1211, 0.7:0.1193, 0.8:0.1175, 0.9:0.1162, 1.0:0.1154}

x_pct    = [r * 100 for r in RATIOS]
gru_vals = df_ratio['rmse'].values
xgb_vals = [xgb_ratio[r] for r in RATIOS]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 좌: ratio별 RMSE 곡선 비교
ax = axes[0]
ax.plot(x_pct, xgb_vals, 'b--o', markersize=5, linewidth=1.8, label='XGB (Static, 4sig×4stat)')
ax.plot(x_pct, gru_vals, 'g-o',  markersize=5, linewidth=2.0, label='GRU-DeltaMeta (Seq, best HP)')
ax.axvline(best_row.ratio * 100, color='green', linestyle=':', linewidth=1.5,
           label=f"GRU best @ {int(best_row.ratio*100)}%  ({best_row.rmse:.4f})")
ax.axhline(0.1153, color='blue', linestyle=':', linewidth=1, alpha=0.6, label='XGB best=0.1153')
ax.set_xlabel('입력 신호 길이 (%)'); ax.set_ylabel('LOCO-CV RMSE (mm)')
ax.set_xticks(x_pct); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title('RMSE vs Input Ratio')

# 우: ratio별 GRU vs XGB 격차 막대
ax2  = axes[1]
gaps = [g - x for g, x in zip(gru_vals, xgb_vals)]
cols = ['#4caf50' if g <= 0 else '#ef5350' for g in gaps]
ax2.bar(x_pct, gaps, color=cols, edgecolor='black', linewidth=0.6)
ax2.axhline(0, color='black', linewidth=1)
ax2.set_xlabel('입력 신호 길이 (%)'); ax2.set_ylabel('ΔRMSE (GRU − XGB)')
ax2.set_xticks(x_pct); ax2.grid(axis='y', alpha=0.3)
ax2.set_title('GRU-DeltaMeta 우위 (음수=GRU 우세)')
for j, (xp, g) in enumerate(zip(x_pct, gaps)):
    sign = '▼' if g <= 0 else '▲'
    ax2.text(xp, g + (0.001 if g >= 0 else -0.002), f'{sign}{abs(g):.4f}',
             ha='center', va='bottom' if g >= 0 else 'top', fontsize=7.5)

plt.suptitle(f'GRU-DeltaMeta vs XGB: RMSE by Input Ratio\n'
             f'(HIDDEN={BEST_H}, LAYERS={BEST_L}, EPOCHS={BEST_E}, LR={BEST_LR:.0e})',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/nasa_milling/gru_ratio_results.png', dpi=150)
plt.show()
print("저장: gru_ratio_results.png")

# ── 수치 테이블 ──────────────────────────────────────────────────────────
print("\n=== ratio별 RMSE 비교 ===")
print(f"  {'ratio':>6} {'GRU-DeltaMeta':>14} {'XGB':>8} {'Δ':>8}")
print("  " + "-"*40)
for row in ratio_results:
    r     = row['ratio']
    delta = row['rmse'] - xgb_ratio[r]
    sign  = '▼' if delta < 0 else '▲'
    print(f"  {r:.1f}    {row['rmse']:>14.4f} {xgb_ratio[r]:>8.4f}  {sign}{abs(delta):.4f}")



  [2단계] 입력 길이 비율 실험
  (HIDDEN=256, LAYERS=3, EPOCHS=200, LR=1e-03)
  ratio=0.1  RMSE=0.2274  (19.8s)
  ratio=0.2  RMSE=0.1373  (20.3s)
  ratio=0.3  RMSE=0.1008  (20.3s)
  ratio=0.4  RMSE=0.1181  (19.8s)
  ratio=0.5  RMSE=0.1101  (21.0s)
  ratio=0.6  RMSE=0.1032  (19.7s)
  ratio=0.7  RMSE=0.0951  (21.3s)
  ratio=0.8  RMSE=0.0918  (19.8s)
  ratio=0.9  RMSE=0.0947  (20.7s)
  ratio=1.0  RMSE=0.0926  (20.2s)

최적 ratio: 0.8  RMSE=0.0918
저장: gru_ratio_results.png

=== ratio별 RMSE 비교 ===
   ratio  GRU-DeltaMeta      XGB        Δ
  ----------------------------------------
  0.1            0.2274   0.1892  ▲0.0382
  0.2            0.1373   0.1507  ▼0.0134
  0.3            0.1008   0.1352  ▼0.0344
  0.4            0.1181   0.1276  ▼0.0095
  0.5            0.1101   0.1221  ▼0.0120
  0.6            0.1032   0.1211  ▼0.0179
  0.7            0.0951   0.1193  ▼0.0242
  0.8            0.0918   0.1175  ▼0.0257
  0.9            0.0947   0.1162  ▼0.0215
  1.0            0.0926   0.1154  ▼0.0228


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 구간 분할: No-load / Entry / Steady / Exit
# Peng et al. (2026) Sec. 3.2 — Unsupervised extraction of steady-state signals
#   Stage 1: No-load rejection (wf from first 3 windows, threshold δ·eps)
#   Stage 2: Active 구간에서 고정 길이(4000 samp) 슬라이딩 윈도우 → min-std 위치 선택
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

# ── 파라미터 ───────────────────────────────────────────────────────────────
SIG_SEG    = 'smcDC'   # 분할 기준 신호
W          = 500        # Stage 1 윈도우 크기 (2 s @ 250 Hz)
STEP       = 250        # Stage 1 스텝 (50% overlap)
DELTA      = 3.0        # Stage 1: no-load 임계 배수 (δ × eps)
STEADY_LEN = 4000       # Stage 2: Steady-cut 고정 길이 (16 s @ 250 Hz)

# ── 구간 분할 함수 ─────────────────────────────────────────────────────────
def segment_pass(signal, W=W, step=STEP, delta=DELTA, steady_len=STEADY_LEN):
    """
    2-Stage unsupervised segmentation (Peng et al. 2026, Sec. 3.2).

    Stage 1 — No-load rejection
      wf  = RMS mean of first 3 windows  (no-load baseline)
      eps = RMS std  of first 3 windows  (noise floor; floor = 1% of wf)
      no-load region: consecutive windows from start where |wi - wf| < delta*eps

    Stage 2 — Steady-cut extraction  (fixed length = steady_len samples)
      Slide a steady_len-sample window through the active (post-no-load) region.
      Select the position with minimum signal std → steady start.
      Slide step = W//2 for efficiency.

    Returns dict: {name: (start_samp, end_samp)}
        'no_load', 'entry', 'steady', 'exit'
    """
    N   = len(signal)
    sig = signal.astype(float)

    # Windowed RMS feature (Stage 1)
    n_wins = max(1, (N - W) // step + 1)
    wf_arr = np.array([
        np.sqrt((sig[k*step : k*step+W] ** 2).mean())
        for k in range(n_wins)
    ])

    # ── Stage 1: No-load 경계 ────────────────────────────────────────────
    wf  = wf_arr[:3].mean()
    eps = max(wf_arr[:3].std(), wf * 0.01, 1e-6)   # floor: 1% of baseline

    end_nl = 0
    for k in range(n_wins):
        if abs(wf_arr[k] - wf) < delta * eps:
            end_nl = k
        else:
            break
    end_nl_samp = min(end_nl * step + W, N)

    # ── Stage 2: Steady-cut (4000 samp, min-std 슬라이딩) ───────────────
    scan_step = W // 2   # 250 sample 단위로 스캔
    avail     = N - end_nl_samp

    if avail < steady_len:
        # active 구간이 steady_len보다 짧으면 전체를 steady로
        return {'no_load': (0, end_nl_samp), 'entry': (end_nl_samp, end_nl_samp),
                'steady':  (end_nl_samp, N),  'exit':  (N, N)}

    best_start = end_nl_samp
    best_std   = float('inf')
    for s in range(end_nl_samp, N - steady_len + 1, scan_step):
        w_std = sig[s : s + steady_len].std()
        if w_std < best_std:
            best_std   = w_std
            best_start = s

    start_steady = best_start
    end_steady   = start_steady + steady_len

    return {
        'no_load': (0,            end_nl_samp),
        'entry':   (end_nl_samp,  start_steady),
        'steady':  (start_steady, end_steady),
        'exit':    (end_steady,   N),
    }

# ── 전체 valid_passes 구간 분할 ────────────────────────────────────────────
all_segs = {}
for item in valid_passes:
    sig = item['p'][SIG_SEG].astype(float)
    all_segs[(item['case'], item['run'])] = segment_pass(sig)

segs_base = {c: all_segs[(c, base_run[c]['run'])] for c in CASES}

print(f"총 {len(all_segs)}개 pass 구간 분할 완료")
print(f"  파라미터: W={W}, step={STEP}, δ={DELTA}, steady_len={STEADY_LEN}")

# ── 대표 케이스 샘플 출력 ───────────────────────────────────────────────────
print(f"\n{'Case':>5} {'Run':>4}  {'No-load':>11}  {'Entry':>11}  {'Steady':>11}  {'Exit':>11}  VB")
print("─" * 75)
for c in CASES:
    items = sorted([x for x in valid_passes if x['case'] == c], key=lambda x: x['run'])
    item  = items[len(items) // 2]
    seg   = all_segs[(c, item['run'])]
    def fmt(s): return f"[{s[0]:4d}~{s[1]:4d}]"
    print(f"  C{c:02d}  r{item['run']:02d}  {fmt(seg['no_load'])}  {fmt(seg['entry'])}  "
          f"{fmt(seg['steady'])}  {fmt(seg['exit'])}  {item['VB']:.3f}")

# ── 구간 길이 통계 ─────────────────────────────────────────────────────────
seg_lens = defaultdict(list)
for seg in all_segs.values():
    for name, (s, e) in seg.items():
        seg_lens[name].append(e - s)

print("\n=== 구간 길이 통계 (samples) ===")
for name in ['no_load', 'entry', 'steady', 'exit']:
    v = np.array(seg_lens[name])
    print(f"  {name:<9}: mean={v.mean():6.0f} ({v.mean()/9000*100:4.1f}%)  "
          f"std={v.std():.0f}  min={v.min():.0f}  max={v.max():.0f}")

# ── 시각화 ─────────────────────────────────────────────────────────────────
SHOW_CASES = [1, 2, 8, 9, 11, 14]
SEG_COLORS = {
    'no_load': '#aaaaaa',
    'entry':   '#4fc3f7',
    'steady':  '#66bb6a',
    'exit':    '#ffa726',
}

def pick_mid_run(c):
    items = sorted([x for x in valid_passes if x['case'] == c], key=lambda x: x['run'])
    return items[len(items) // 2]

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, c in zip(axes.flat, SHOW_CASES):
    item = pick_mid_run(c)
    sig  = item['p'][SIG_SEG].astype(float)
    seg  = all_segs[(c, item['run'])]
    t    = np.arange(len(sig)) / 250.0

    ax.plot(t, sig, color='#333333', linewidth=0.5, alpha=0.85, zorder=2)
    for sname, (s0, e0) in seg.items():
        if e0 > s0:
            ax.axvspan(s0/250, e0/250, alpha=0.28,
                       color=SEG_COLORS[sname], zorder=1)
        if e0 > s0 and sname != 'no_load':
            ax.axvline(s0/250, color=SEG_COLORS[sname], linewidth=1.2,
                       linestyle='--', alpha=0.9, zorder=3)

    # Steady 구간 std 표시
    s0, e0 = seg['steady']
    st_std = sig[s0:e0].std() if e0 > s0 else 0
    ax.set_title(f'Case {c:02d}  run {item["run"]:02d}  VB={item["VB"]:.3f} mm'
                 f'  (steady std={st_std:.2f})', fontsize=9.5)
    ax.set_xlim(0, len(sig)/250)
    ax.grid(alpha=0.2)

    parts = [f"{k[0].upper()}{k[1]}:{e-s}" for k, (s, e) in seg.items()]
    ax.set_xlabel(f"Time (s)   [{'  '.join(parts)}]", fontsize=7.5)
    ax.set_ylabel('smcAC')

patches = [mpatches.Patch(color=c, alpha=0.5, label=n)
           for n, c in SEG_COLORS.items()]
fig.legend(handles=patches, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.01))

plt.suptitle(
    f'Segmentation: No-load / Entry / Steady({STEADY_LEN}samp) / Exit\n'
    f'Peng et al. (2026) Sec. 3.2 — Stage2: min-std sliding window',
    fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.06, 1, 1])

save_path = '/content/drive/MyDrive/dataset/nasa_milling/segmentation_viz.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n저장: {save_path}")
print(f"\n다음 셀에서 사용 가능한 변수:")
print(f"  all_segs   : dict{{(case,run): {{name:(start,end)}}}}  — {len(all_segs)} entries")
print(f"  segs_base  : dict{{case: {{name:(start,end)}}}}  — {len(segs_base)} cases")

총 164개 pass 구간 분할 완료
  파라미터: W=500, step=250, δ=3.0, steady_len=4000

 Case  Run      No-load        Entry       Steady         Exit  VB
───────────────────────────────────────────────────────────────────────────
  C01  r09  [   0~1000]  [1000~2250]  [2250~6250]  [6250~9000]  0.280
  C02  r08  [   0~1000]  [1000~3250]  [3250~7250]  [7250~9000]  0.220
  C03  r09  [   0~2000]  [2000~5000]  [5000~9000]  [9000~9000]  0.230
  C04  r04  [   0~1000]  [1000~5000]  [5000~9000]  [9000~9000]  0.310
  C05  r04  [   0~1250]  [1250~1250]  [1250~5250]  [5250~9000]  0.440
  C06  r01  [   0~1000]  [1000~5000]  [5000~9000]  [9000~9000]  0.000
  C07  r04  [   0~1000]  [1000~3250]  [3250~7250]  [7250~9000]  0.220
  C08  r04  [   0~1250]  [1250~2500]  [2500~6500]  [6500~9000]  0.370
  C09  r05  [   0~1000]  [1000~2000]  [2000~6000]  [6000~9000]  0.270
  C10  r06  [   0~1000]  [1000~4750]  [4750~8750]  [8750~9000]  0.360
  C11  r12  [   0~1000]  [1000~5000]  [5000~9000]  [9000~9000]  0.230
  C12  r09  [   0

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Exit 식별 검증: Reverse Stage 1 segmentation
# 기준 센서: smcDC
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

SIG_SEG     = 'smcDC'
W           = 50
STEP        = 25
DELTA       = 1.0
STEADY_LEN  = 4000
EXIT_THRESH = STEADY_LEN // 2   # 2000

def stage1_nl_end(signal, W=W, step=STEP, delta=DELTA):
    N   = len(signal)
    sig = signal.astype(float)
    n_w = max(1, (N - W) // step + 1)
    rms = np.array([np.sqrt((sig[k*step:k*step+W]**2).mean()) for k in range(n_w)])
    wf  = rms[:3].mean()
    eps = max(rms[:3].std(), wf * 0.01, 1e-6)
    end = 0
    for k in range(n_w):
        if abs(rms[k] - wf) < delta * eps:
            end = k
        else:
            break
    return min(end * step + W, N)

def segment_pass_v2(signal, W=W, step=STEP, delta=DELTA, steady_len=STEADY_LEN):
    N   = len(signal)
    sig = signal.astype(float)

    nl_end      = stage1_nl_end(sig,       W, step, delta)
    rev_nl_len  = stage1_nl_end(sig[::-1], W, step, delta)
    trail_start = N - rev_nl_len
    trail_start = max(trail_start, nl_end + steady_len)
    trail_start = min(trail_start, N)

    avail = trail_start - nl_end
    if avail < steady_len:
        ss = nl_end
    else:
        best_s, best_std = nl_end, float('inf')
        for s in range(nl_end, trail_start - steady_len + 1, W // 2):
            v = sig[s:s + steady_len].std()
            if v < best_std:
                best_std, best_s = v, s
        ss = best_s

    se = min(ss + steady_len, N)

    return {
        'no_load':     (0,      nl_end),
        'entry':       (nl_end, ss),
        'steady':      (ss,     se),
        'exit':        (se,     N),
        '_rev_nl_len': rev_nl_len,
    }

# ── 전체 pass 분할 ──────────────────────────────────────────────────────────
all_segs_v2 = {}
for item in valid_passes:
    sig = item['p'][SIG_SEG].astype(float)
    all_segs_v2[(item['case'], item['run'])] = segment_pass_v2(sig)

rev_nl_lens = {k: v['_rev_nl_len'] for k, v in all_segs_v2.items()}
has_exit    = {k: v < EXIT_THRESH  for k, v in rev_nl_lens.items()}
n_exit      = sum(has_exit.values())

print(f"분할 완료: {len(all_segs_v2)} passes  (SIG={SIG_SEG}, δ={DELTA}, steady={STEADY_LEN})")
print(f"Exit 판별 기준: rev_nl < {EXIT_THRESH} samples")
print(f"  Exit 있음: {n_exit}개   Exit 없음: {len(has_exit)-n_exit}개")

print(f"\n{'Case':>5}  Exit / 전체   Exit 있는 run 목록")
print("─" * 65)
no_exit_cases = []
for c in CASES:
    runs = [(c, item['run']) for item in valid_passes if item['case'] == c]
    n_e  = sum(has_exit[k] for k in runs)
    detail = ', '.join(f"r{k[1]}" for k in runs if has_exit[k]) or '(없음)'
    print(f"  C{c:02d}   {n_e:2d} / {len(runs):2d}   {detail}")
    if n_e == 0:
        no_exit_cases.append(c)

print(f"\nExit 없는 케이스: {no_exit_cases}")

seg_lens_v2 = defaultdict(list)
for seg in all_segs_v2.values():
    for nm in ['no_load', 'entry', 'steady', 'exit']:
        s, e = seg[nm]; seg_lens_v2[nm].append(e - s)

print("\n=== 구간 길이 통계 v2 (samples) ===")
for nm in ['no_load', 'entry', 'steady', 'exit']:
    v = np.array(seg_lens_v2[nm])
    print(f"  {nm:<9}: mean={v.mean():6.0f} ({v.mean()/9000*100:4.1f}%)"
          f"  std={v.std():.0f}  min={v.min():.0f}  max={v.max():.0f}")

# ── 케이스별 대표 run 선택 (중간 run) ──────────────────────────────────────
def pick_mid(c):
    items = sorted([x for x in valid_passes if x['case'] == c], key=lambda x: x['run'])
    return items[len(items) // 2]

SEG_COLORS = {'no_load':'#aaaaaa','entry':'#4fc3f7','steady':'#66bb6a','exit':'#ffa726'}

# ══════════════════════════════════════════════════════════════════════════
# 시각화 1: 전체 16개 케이스 × 순방향 신호 + 4구간 (4×4)
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(4, 4, figsize=(22, 16))
for ax, c in zip(axes1.flat, CASES):
    item = pick_mid(c)
    sig  = item['p'][SIG_SEG].astype(float)
    seg  = all_segs_v2[(c, item['run'])]
    t    = np.arange(len(sig)) / 250.0

    ax.plot(t, sig, color='#333333', lw=0.5, alpha=0.85, zorder=2)
    for nm in ['no_load', 'entry', 'steady', 'exit']:
        s0, e0 = seg[nm]
        if e0 > s0:
            ax.axvspan(s0/250, e0/250, alpha=0.28, color=SEG_COLORS[nm], zorder=1)
        if e0 > s0 and nm != 'no_load':
            ax.axvline(s0/250, color=SEG_COLORS[nm], lw=1.2, ls='--', alpha=0.9, zorder=3)

    rev_nl = seg['_rev_nl_len']
    flag   = '[Exit O]' if has_exit[(c, item['run'])] else '[Exit X]'
    ax.set_title(f'C{c:02d} r{item["run"]:02d}  VB={item["VB"]:.3f}  {flag}', fontsize=8.5)
    ax.set_xlim(0, len(sig)/250); ax.grid(alpha=0.2)
    parts = [f"{nm[0].upper()}:{seg[nm][1]-seg[nm][0]}" for nm in ['no_load','entry','steady','exit']]
    ax.set_xlabel(' '.join(parts), fontsize=7)
    ax.set_ylabel(SIG_SEG, fontsize=7)
    ax.tick_params(labelsize=7)

patches = [mpatches.Patch(color=c, alpha=0.5, label=n) for n, c in SEG_COLORS.items()]
fig1.legend(handles=patches, loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.005))
plt.suptitle(f'Segmentation v2 — {SIG_SEG}  (W={W}, δ={DELTA}, steady={STEADY_LEN}, thresh={EXIT_THRESH})\n'
             f'케이스별 중간 run 대표  |  [Exit O]=Exit 있음  [Exit X]=Exit 없음',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
p1 = '/content/drive/MyDrive/dataset/nasa_milling/seg_v2_all_cases.png'
plt.savefig(p1, dpi=150, bbox_inches='tight'); plt.show()
print(f"저장: {p1}")

# ══════════════════════════════════════════════════════════════════════════
# 시각화 2: 역순 no-load 길이 분포 (전체 pass)
# ══════════════════════════════════════════════════════════════════════════
ordered_keys = []
for c in CASES:
    for item in sorted([x for x in valid_passes if x['case'] == c], key=lambda x: x['run']):
        ordered_keys.append((c, item['run']))

bar_vals   = [rev_nl_lens[k] for k in ordered_keys]
bar_colors = ['#ef5350' if has_exit[k] else '#78909c' for k in ordered_keys]

fig2, ax2 = plt.subplots(figsize=(18, 4))
bx = np.arange(len(ordered_keys))
ax2.bar(bx, bar_vals, color=bar_colors, width=0.8, edgecolor='none', alpha=0.85)
ax2.axhline(EXIT_THRESH, color='black', ls='--', lw=1.5,
            label=f'Exit 판별 임계값 ({EXIT_THRESH} samp)')
tick = 0
for c in CASES:
    n = sum(1 for x in valid_passes if x['case'] == c)
    ax2.axvline(tick - 0.5, color='black', lw=0.6, alpha=0.4)
    ax2.text(tick + n/2 - 0.5, max(bar_vals) * 1.02, f'C{c:02d}',
             ha='center', fontsize=7, fontweight='bold')
    tick += n
ax2.set_ylabel('역순 no-load 길이 (samples)')
ax2.set_title('Pass별 역순 no-load 길이  |  빨강=Exit 있음  회색=Exit 없음')
ax2.set_xticks(bx[::2])
ax2.set_xticklabels([f"r{k[1]}" for k in ordered_keys[::2]], rotation=60, fontsize=6.5)
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
p2 = '/content/drive/MyDrive/dataset/nasa_milling/seg_v2_rev_nl_dist.png'
plt.savefig(p2, dpi=150, bbox_inches='tight'); plt.show()
print(f"저장: {p2}")

# ══════════════════════════════════════════════════════════════════════════
# 시각화 3: 전체 16개 케이스 × 역순 신호 + Stage 1 탐지 (4×4)
# ══════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(4, 4, figsize=(22, 16))
for ax, c in zip(axes3.flat, CASES):
    item    = pick_mid(c)
    sig     = item['p'][SIG_SEG].astype(float)
    sig_rev = sig[::-1]
    t       = np.arange(len(sig_rev)) / 250.0
    rev_nl  = all_segs_v2[(c, item['run'])]['_rev_nl_len']

    ax.plot(t, sig_rev, color='#555555', lw=0.5, alpha=0.85)
    ax.axvspan(0, rev_nl/250, alpha=0.3, color='#aaaaaa')
    ax.axvline(rev_nl/250, color='#ef5350', lw=1.3, ls='--',
               label=f'nl_end={rev_nl}')
    ax.axhline(sig_rev[:3*W].mean(), color='navy', lw=0.8, ls=':', alpha=0.6)

    flag = '[Exit O]' if has_exit[(c, item['run'])] else '[Exit X]'
    ax.set_title(f'C{c:02d} r{item["run"]:02d}  rev_nl={rev_nl}  {flag}', fontsize=8.5)
    ax.set_xlim(0, len(sig_rev)/250); ax.grid(alpha=0.2)
    ax.set_xlabel('Time in reversed signal (s)', fontsize=7)
    ax.set_ylabel(f'{SIG_SEG} (rev)', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle(f'역순(Reversed) {SIG_SEG} 신호 + Stage 1 탐지  (임계={EXIT_THRESH})\n'
             f'회색=역순 no-load 구간  빨간선=no-load 끝  파란점선=wf(rev)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
p3 = '/content/drive/MyDrive/dataset/nasa_milling/seg_v2_reversed_signal.png'
plt.savefig(p3, dpi=150, bbox_inches='tight'); plt.show()
print(f"저장: {p3}")

print(f"\n다음 셀에서 사용 가능한 변수:")
print(f"  all_segs_v2 : {{(case,run): segs + '_rev_nl_len'}}  — {len(all_segs_v2)} entries")
print(f"  has_exit    : {{(case,run): bool}}  — {n_exit}개 Exit 있음")
print(f"  EXIT_THRESH = {EXIT_THRESH}")

분할 완료: 164 passes  (SIG=smcDC, δ=1.0, steady=4000)
Exit 판별 기준: rev_nl < 2000 samples
  Exit 있음: 157개   Exit 없음: 7개

 Case  Exit / 전체   Exit 있는 run 목록
─────────────────────────────────────────────────────────────────
  C01   17 / 17   r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13, r14, r15, r16, r17
  C02   13 / 13   r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13, r14
  C03   14 / 14   r1, r2, r3, r5, r6, r7, r8, r9, r10, r11, r12, r14, r15, r16
  C04    7 /  7   r1, r2, r3, r4, r5, r6, r7
  C05    6 /  6   r1, r2, r3, r4, r5, r6
  C06    1 /  1   r1
  C07    7 /  7   r1, r2, r3, r4, r5, r6, r7
  C08    6 /  6   r1, r2, r3, r4, r5, r6
  C09    9 /  9   r1, r2, r3, r4, r5, r6, r7, r8, r9
  C10    7 / 10   r1, r2, r3, r4, r5, r6, r7
  C11   23 / 23   r1, r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13, r14, r15, r16, r17, r18, r19, r20, r21, r22, r23
  C12   14 / 14   r2, r3, r4, r5, r6, r7, r8, r9, r10, r11, r12, r13, r14, r15
  C13   13 / 15   r1, r2, r3, r4, r5, r6, r7,

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# No-load 제외 + Active Ratio 실험
# Stage 1(smcDC, W=500, δ=3.0)으로 no-load end 탐지 →
# 이후 active 구간만 ratio 비율로 잘라 피처 추출 → LOCO-CV
# 비교: 기존 전체신호 ratio(XGB) vs no-load 제외 ratio(Ridge/RF/XGB/SVR)
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np, warnings, time, copy
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
np.random.seed(42)

SIG_NL   = 'smcDC'
W_NL     = 50
STEP_NL  = 25
DELTA_NL = 1.0
SIG_COLS = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
RATIOS   = [r / 10 for r in range(1, 11)]
N_FULL   = 9000

def stage1_nl_end(signal, W=W_NL, step=STEP_NL, delta=DELTA_NL):
    N   = len(signal)
    sig = signal.astype(float)
    n_w = max(1, (N - W) // step + 1)
    rms = np.array([np.sqrt((sig[k*step:k*step+W]**2).mean()) for k in range(n_w)])
    wf  = rms[:3].mean()
    eps = max(rms[:3].std(), wf * 0.01, 1e-6)
    end = 0
    for k in range(n_w):
        if abs(rms[k] - wf) < delta * eps:
            end = k
        else:
            break
    return min(end * step + W, N)

nl_ends = {(it['case'], it['run']): stage1_nl_end(it['p'][SIG_NL]) for it in valid_passes}
base_nl = {c: nl_ends[(c, base_run[c]['run'])] for c in CASES}

nl_vals = list(nl_ends.values())
print(f"No-load end 분포: mean={np.mean(nl_vals):.0f}  std={np.std(nl_vals):.0f}"
      f"  min={np.min(nl_vals)}  max={np.max(nl_vals)}")

extreme = [(k, v) for k, v in nl_ends.items() if v >= N_FULL - 100]
if extreme:
    print(f"  ※ nl_end >= {N_FULL-100}: {extreme}  -> nl_start을 N-100으로 클램핑")

# ── 피처 추출 (no-load 제외 active 구간 × ratio) ───────────────────────────
def extract_active(p, nl_start, ratio, N=N_FULL):
    nl_s       = min(nl_start, N - 100)   # active 구간 최소 100샘플 보장
    active_len = N - nl_s
    n          = max(1, int(active_len * ratio))
    feats = []
    for col in SIG_COLS:
        s = p[col][nl_s : nl_s + n].astype(float)
        if len(s) == 0:
            s = p[col][-1:].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── LOCO-CV ─────────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

results = {m: [] for m in MODELS}
t0 = time.time()

print(f"\n{'Ratio':>6}", end='')
for m in MODELS: print(f"  {m:>8}", end='')
print()
print("-" * 50)

for ratio in RATIOS:
    X_list, y_list, cas_list = [], [], []
    for item in valid_passes:
        c      = item['case']
        nl_s   = nl_ends[(c, item['run'])]
        nl_b   = base_nl[c]
        f_raw  = extract_active(item['p'],        nl_s, ratio)
        f_base = extract_active(base_run[c]['p'],  nl_b, ratio)
        meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
        X_list.append(np.concatenate([f_raw, f_raw - f_base, meta]))
        y_list.append(item['VB'])
        cas_list.append(c)

    X   = np.array(X_list)
    y   = np.array(y_list)
    cas = np.array(cas_list)

    print(f"  {ratio:.1f} ", end='', flush=True)
    for mname, mproto in MODELS.items():
        rmses = []
        for tc in CASES:
            tr = cas != tc; te = cas == tc
            sc  = StandardScaler()
            Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
            m   = copy.deepcopy(mproto)
            m.fit(Xtr, y[tr])
            rmses.append(np.sqrt(mean_squared_error(y[te], m.predict(Xte))))
        r = float(np.mean(rmses))
        results[mname].append(r)
        print(f"  {r:>8.4f}", end='', flush=True)
    print()

print(f"\n총 소요: {time.time()-t0:.1f}s")

print("\n=== 모델별 최적 ratio (No-load 제외) ===")
for mname, vals in results.items():
    best_i = int(np.argmin(vals))
    print(f"  {mname:<6}: ratio={RATIOS[best_i]:.1f}  RMSE={vals[best_i]:.4f}")

# 기존 전체신호 XGB ratio 결과 (parameter_search.md 기준)
xgb_full = {0.1:0.1851, 0.2:0.1724, 0.3:0.1300, 0.4:0.1242, 0.5:0.1172,
            0.6:0.1146, 0.7:0.1157, 0.8:0.1190, 0.9:0.1161, 1.0:0.1153}

x_pct = [r * 100 for r in RATIOS]
clr   = {'Ridge':'#9e9e9e', 'RF':'#43a047', 'XGB':'#e65100', 'SVR':'#1565c0'}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for mname, vals in results.items():
    best_i = int(np.argmin(vals))
    ax.plot(x_pct, vals, '-o', ms=5, lw=1.8, color=clr[mname],
            label=f'{mname} NL-excl (best={vals[best_i]:.4f}@{int(RATIOS[best_i]*100)}%)')
ax.plot(x_pct, [xgb_full[r] for r in RATIOS], 'k--s', ms=5, lw=1.5,
        label=f'XGB full-signal best={min(xgb_full.values()):.4f}@60%')
ax.axhline(min(xgb_full.values()), color='black', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('Active 신호 사용 비율 (%)'); ax.set_ylabel('LOCO-CV RMSE (mm)')
ax.set_title('No-load 제외 Active Ratio vs 전체신호 Ratio')
ax.set_xticks(x_pct); ax.legend(fontsize=8.5); ax.grid(alpha=0.3)

ax2 = axes[1]
xgb_new = results['XGB']
gaps    = [n - xgb_full[r] for n, r in zip(xgb_new, RATIOS)]
cols    = ['#43a047' if g <= 0 else '#ef5350' for g in gaps]
ax2.bar(x_pct, gaps, color=cols, edgecolor='black', lw=0.6)
ax2.axhline(0, color='black', lw=1)
ax2.set_xlabel('Active 신호 사용 비율 (%)')
ax2.set_ylabel('DELTA_RMSE (NL-excl XGB - Full XGB)')
ax2.set_title('XGB: No-load 제외 vs 전체신호\n(음수 = No-load 제외가 유리)')
ax2.set_xticks(x_pct); ax2.grid(axis='y', alpha=0.3)
for xp, g in zip(x_pct, gaps):
    sign = 'v' if g <= 0 else '^'
    va   = 'bottom' if g >= 0 else 'top'
    off  = 0.0005 if g >= 0 else -0.0005
    ax2.text(xp, g + off, f'{sign}{abs(g):.4f}', ha='center', va=va, fontsize=7.5)

plt.suptitle('No-load 제외 Active Ratio 실험  (Raw+Delta+Meta, 4sig×4stat, LOCO-CV)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/noload_excl_ratio.png'
plt.savefig(save_path, dpi=150); plt.show()
print(f"\n저장: {save_path}")

print(f"\n{'Ratio':>6}  {'Ridge':>8}  {'RF':>8}  {'XGB(NL)':>9}  {'SVR':>8}  {'XGB(Full)':>10}  {'D(XGB)':>8}")
print("-" * 72)
for i, ratio in enumerate(RATIOS):
    d    = results['XGB'][i] - xgb_full[ratio]
    sign = 'v' if d <= 0 else '^'
    print(f"  {ratio:.1f}   {results['Ridge'][i]:>8.4f}  {results['RF'][i]:>8.4f}"
          f"  {results['XGB'][i]:>9.4f}  {results['SVR'][i]:>8.4f}"
          f"  {xgb_full[ratio]:>10.4f}  {sign}{abs(d):.4f}")

No-load end 분포: mean=623  std=294  min=50  max=2050

 Ratio     Ridge        RF       XGB       SVR
--------------------------------------------------
  0.1     0.2140    0.1795    0.1640    0.1546
  0.2     0.1497    0.1571    0.1555    0.1557
  0.3     0.1749    0.1323    0.1280    0.1427
  0.4     0.1576    0.1250    0.1237    0.1429
  0.5     0.1784    0.1248    0.1292    0.1398
  0.6     0.1719    0.1182    0.1180    0.1321
  0.7     0.1680    0.1186    0.1226    0.1251
  0.8     0.1617    0.1166    0.1221    0.1208
  0.9     0.1544    0.1180    0.1178    0.1223
  1.0     0.1601    0.1152    0.1177    0.1188

총 소요: 171.9s

=== 모델별 최적 ratio (No-load 제외) ===
  Ridge : ratio=0.2  RMSE=0.1497
  RF    : ratio=1.0  RMSE=0.1152
  XGB   : ratio=1.0  RMSE=0.1177
  SVR   : ratio=1.0  RMSE=0.1188

저장: /content/drive/MyDrive/dataset/nasa_milling/noload_excl_ratio.png

 Ratio     Ridge        RF    XGB(NL)       SVR   XGB(Full)    D(XGB)
--------------------------------------------------------

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Steady-only Input Ratio 실험  (W=50, STEP=25, DELTA=1.0)
# segment_pass_v2를 W=50/STEP=25/DELTA=1.0으로 재계산 →
# Steady 구간 내 ratio 비율만큼 잘라 피처 추출 → LOCO-CV
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np, warnings, time, copy
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
np.random.seed(42)

W          = 50
STEP       = 25
DELTA      = 1.0
STEADY_LEN = 4000
SIG_SEG    = 'smcDC'
SIG_COLS   = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
RATIOS     = [r / 10 for r in range(1, 11)]
N_FULL     = 9000

# ── 세그멘테이션 v2 (파라미터 주입 버전) ──────────────────────────────────
def _nl_end(signal, W, step, delta):
    N   = len(signal); sig = signal.astype(float)
    n_w = max(1, (N - W) // step + 1)
    rms = np.array([np.sqrt((sig[k*step:k*step+W]**2).mean()) for k in range(n_w)])
    wf  = rms[:3].mean()
    eps = max(rms[:3].std(), wf * 0.01, 1e-6)
    end = 0
    for k in range(n_w):
        if abs(rms[k] - wf) < delta * eps: end = k
        else: break
    return min(end * step + W, N)

def seg_v2(signal, W, step, delta, steady_len):
    N = len(signal); sig = signal.astype(float)
    nl_end      = _nl_end(sig,       W, step, delta)
    rev_nl_len  = _nl_end(sig[::-1], W, step, delta)
    trail_start = max(N - rev_nl_len, nl_end + steady_len)
    trail_start = min(trail_start, N)
    avail = trail_start - nl_end
    if avail < steady_len:
        ss = nl_end
    else:
        best_s, best_std = nl_end, float('inf')
        for s in range(nl_end, trail_start - steady_len + 1, W // 2):
            v = sig[s : s + steady_len].std()
            if v < best_std: best_std, best_s = v, s
        ss = best_s
    se = min(ss + steady_len, N)
    return {'no_load': (0, nl_end), 'entry': (nl_end, ss),
            'steady':  (ss, se),    'exit':   (se, N)}

print(f"세그멘테이션 재계산 중... (W={W}, STEP={STEP}, DELTA={DELTA})")
t0 = time.time()
all_segs_s = {}
for item in valid_passes:
    sig = item['p'][SIG_SEG].astype(float)
    all_segs_s[(item['case'], item['run'])] = seg_v2(sig, W, STEP, DELTA, STEADY_LEN)
print(f"완료: {time.time()-t0:.1f}s")

sl = [v['steady'][1] - v['steady'][0] for v in all_segs_s.values()]
print(f"Steady 길이: mean={np.mean(sl):.0f}  std={np.std(sl):.0f}"
      f"  min={np.min(sl)}  max={np.max(sl)}")
n_short = sum(1 for l in sl if l < STEADY_LEN)
if n_short:
    print(f"  ※ Steady < {STEADY_LEN} 샘플인 패스: {n_short}개")

base_segs_s = {c: all_segs_s[(c, base_run[c]['run'])] for c in CASES}

# ── 피처 추출 ─────────────────────────────────────────────────────────────
def extract_steady(p, segs, ratio):
    ss, se = segs['steady']
    avail  = se - ss
    n      = min(max(1, int(STEADY_LEN * ratio)), avail)
    feats  = []
    for col in SIG_COLS:
        s = p[col][ss : ss + n].astype(float)
        if len(s) == 0: s = p[col][ss : ss+1].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── LOCO-CV ──────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

results_s = {m: [] for m in MODELS}
t1 = time.time()

print(f"\n{'Ratio':>6}", end='')
for m in MODELS: print(f"  {m:>8}", end='')
print()
print("-" * 50)

for ratio in RATIOS:
    X_list, y_list, cas_list = [], [], []
    for item in valid_passes:
        c      = item['case']
        segs   = all_segs_s[(c, item['run'])]
        segs_b = base_segs_s[c]
        f_raw  = extract_steady(item['p'],        segs,   ratio)
        f_base = extract_steady(base_run[c]['p'],  segs_b, ratio)
        meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
        X_list.append(np.concatenate([f_raw, f_raw - f_base, meta]))
        y_list.append(item['VB'])
        cas_list.append(c)

    X   = np.array(X_list)
    y   = np.array(y_list)
    cas = np.array(cas_list)

    print(f"  {ratio:.1f} ", end='', flush=True)
    for mname, mproto in MODELS.items():
        rmses = []
        for tc in CASES:
            tr = cas != tc; te = cas == tc
            sc  = StandardScaler()
            Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
            m   = copy.deepcopy(mproto)
            m.fit(Xtr, y[tr])
            rmses.append(np.sqrt(mean_squared_error(y[te], m.predict(Xte))))
        r = float(np.mean(rmses))
        results_s[mname].append(r)
        print(f"  {r:>8.4f}", end='', flush=True)
    print()

print(f"\n총 소요: {time.time()-t1:.1f}s")

print("\n=== 모델별 최적 ratio (Steady only) ===")
for mname, vals in results_s.items():
    best_i = int(np.argmin(vals))
    print(f"  {mname:<6}: ratio={RATIOS[best_i]:.1f}  RMSE={vals[best_i]:.4f}")

# ── 시각화 ──────────────────────────────────────────────────────────────────
xgb_full = {0.1:0.1851, 0.2:0.1724, 0.3:0.1300, 0.4:0.1242, 0.5:0.1172,
            0.6:0.1146, 0.7:0.1157, 0.8:0.1190, 0.9:0.1161, 1.0:0.1153}

x_pct = [r * 100 for r in RATIOS]
clr   = {'Ridge':'#9e9e9e', 'RF':'#43a047', 'XGB':'#e65100', 'SVR':'#1565c0'}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for mname, vals in results_s.items():
    best_i = int(np.argmin(vals))
    ax.plot(x_pct, vals, '-o', ms=5, lw=1.8, color=clr[mname],
            label=f'{mname} Steady (best={vals[best_i]:.4f}@{int(RATIOS[best_i]*100)}%)')
ax.plot(x_pct, [xgb_full[r] for r in RATIOS], 'k--s', ms=5, lw=1.5,
        label='XGB Full-signal best=0.1146@60%')
ax.axhline(0.1146, color='black', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('Steady 구간 사용 비율 (%)'); ax.set_ylabel('LOCO-CV RMSE (mm)')
ax.set_title(f'Steady-only Ratio  (W={W}/STEP={STEP}/DELTA={DELTA})')
ax.set_xticks(x_pct); ax.legend(fontsize=8.5); ax.grid(alpha=0.3)

ax2 = axes[1]
xgb_st = results_s['XGB']
gaps   = [n - xgb_full[r] for n, r in zip(xgb_st, RATIOS)]
cols   = ['#43a047' if g <= 0 else '#ef5350' for g in gaps]
ax2.bar(x_pct, gaps, color=cols, edgecolor='black', lw=0.6)
ax2.axhline(0, color='black', lw=1)
ax2.set_xlabel('Steady 구간 사용 비율 (%)')
ax2.set_ylabel('DELTA_RMSE (Steady XGB - Full XGB)')
ax2.set_title('XGB: Steady-only vs 전체신호\n(음수 = Steady가 유리)')
ax2.set_xticks(x_pct); ax2.grid(axis='y', alpha=0.3)
for xp, g in zip(x_pct, gaps):
    sign = 'v' if g <= 0 else '^'
    va   = 'bottom' if g >= 0 else 'top'
    off  = 0.0005 if g >= 0 else -0.0005
    ax2.text(xp, g + off, f'{sign}{abs(g):.4f}', ha='center', va=va, fontsize=7.5)

plt.suptitle(f'Steady-only Input Ratio  (W={W}/STEP={STEP}/DELTA={DELTA}, Raw+Delta+Meta, LOCO-CV)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/steady_ratio.png'
plt.savefig(save_path, dpi=150); plt.show()
print(f"\n저장: {save_path}")

print(f"\n{'Ratio':>6}  {'Ridge':>8}  {'RF':>8}  {'XGB(St)':>9}  {'SVR':>8}  {'XGB(Full)':>10}  {'D(XGB)':>8}")
print("-" * 72)
for i, ratio in enumerate(RATIOS):
    d    = results_s['XGB'][i] - xgb_full[ratio]
    sign = 'v' if d <= 0 else '^'
    print(f"  {ratio:.1f}   {results_s['Ridge'][i]:>8.4f}  {results_s['RF'][i]:>8.4f}"
          f"  {results_s['XGB'][i]:>9.4f}  {results_s['SVR'][i]:>8.4f}"
          f"  {xgb_full[ratio]:>10.4f}  {sign}{abs(d):.4f}")

세그멘테이션 재계산 중... (W=50, STEP=25, DELTA=1.0)
완료: 4.1s
Steady 길이: mean=4000  std=0  min=4000  max=4000

 Ratio     Ridge        RF       XGB       SVR
--------------------------------------------------
  0.1     0.2067    0.1393    0.1367    0.1331
  0.2     0.1766    0.1395    0.1447    0.1443
  0.3     0.1644    0.1340    0.1344    0.1400
  0.4     0.1646    0.1202    0.1143    0.1349
  0.5     0.1723    0.1302    0.1198    0.1423
  0.6     0.1711    0.1312    0.1216    0.1361
  0.7     0.1688    0.1290    0.1232    0.1380
  0.8     0.1644    0.1257    0.1235    0.1347
  0.9     0.1691    0.1255    0.1164    0.1335
  1.0     0.1774    0.1242    0.1211    0.1369

총 소요: 175.2s

=== 모델별 최적 ratio (Steady only) ===
  Ridge : ratio=0.8  RMSE=0.1644
  RF    : ratio=0.4  RMSE=0.1202
  XGB   : ratio=0.4  RMSE=0.1143
  SVR   : ratio=0.1  RMSE=0.1331

저장: /content/drive/MyDrive/dataset/nasa_milling/steady_ratio.png

 Ratio     Ridge        RF    XGB(St)       SVR   XGB(Full)    D(XGB)
------------

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Entry+Steady Input Ratio 실험  (W=50, STEP=25, DELTA=1.0)
# all_segs_s 재사용 → Entry+Steady 구간 [nl_end, se] 내 ratio만큼 잘라 피처 추출
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np, warnings, time, copy
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
np.random.seed(42)

W          = 50
STEP       = 25
DELTA      = 1.0
STEADY_LEN = 4000
SIG_SEG    = 'smcDC'
SIG_COLS   = ['smcAC', 'smcDC', 'AE_table', 'AE_spindle']
RATIOS     = [r / 10 for r in range(1, 11)]
N_FULL     = 9000

# all_segs_s가 없으면 재계산 (이전 셀 의존)
try:
    _ = all_segs_s
    print(f"all_segs_s 재사용 ({len(all_segs_s)}개 패스, W={W}/STEP={STEP}/DELTA={DELTA})")
except NameError:
    def _nl_end(signal, W, step, delta):
        N   = len(signal); sig = signal.astype(float)
        n_w = max(1, (N - W) // step + 1)
        rms = np.array([np.sqrt((sig[k*step:k*step+W]**2).mean()) for k in range(n_w)])
        wf  = rms[:3].mean()
        eps = max(rms[:3].std(), wf * 0.01, 1e-6)
        end = 0
        for k in range(n_w):
            if abs(rms[k] - wf) < delta * eps: end = k
            else: break
        return min(end * step + W, N)

    def seg_v2(signal, W, step, delta, steady_len):
        N = len(signal); sig = signal.astype(float)
        nl_end     = _nl_end(sig,       W, step, delta)
        rev_nl_len = _nl_end(sig[::-1], W, step, delta)
        trail_start = max(N - rev_nl_len, nl_end + steady_len)
        trail_start = min(trail_start, N)
        avail = trail_start - nl_end
        if avail < steady_len:
            ss = nl_end
        else:
            best_s, best_std = nl_end, float('inf')
            for s in range(nl_end, trail_start - steady_len + 1, W // 2):
                v = sig[s : s + steady_len].std()
                if v < best_std: best_std, best_s = v, s
            ss = best_s
        se = min(ss + steady_len, N)
        return {'no_load': (0, nl_end), 'entry': (nl_end, ss),
                'steady':  (ss, se),    'exit':   (se, N)}

    print("all_segs_s 재계산 중...")
    t0 = time.time()
    all_segs_s = {}
    for item in valid_passes:
        sig = item['p'][SIG_SEG].astype(float)
        all_segs_s[(item['case'], item['run'])] = seg_v2(sig, W, STEP, DELTA, STEADY_LEN)
    print(f"완료: {time.time()-t0:.1f}s")

base_segs_s = {c: all_segs_s[(c, base_run[c]['run'])] for c in CASES}

# Entry+Steady 구간 = [nl_end, se]
es_lens = []
for v in all_segs_s.values():
    nl_end = v['no_load'][1]
    se     = v['steady'][1]
    es_lens.append(se - nl_end)
print(f"Entry+Steady 길이: mean={np.mean(es_lens):.0f}  std={np.std(es_lens):.0f}"
      f"  min={np.min(es_lens)}  max={np.max(es_lens)}")

# ── 피처 추출 (Entry+Steady 구간 × ratio) ─────────────────────────────────
def extract_es(p, segs, ratio):
    nl_end = segs['no_load'][1]
    se     = segs['steady'][1]
    total  = se - nl_end          # Entry+Steady 전체 길이
    n      = max(1, int(total * ratio))
    feats  = []
    for col in SIG_COLS:
        s = p[col][nl_end : nl_end + n].astype(float)
        if len(s) == 0: s = p[col][nl_end : nl_end+1].astype(float)
        feats += [s.mean(), s.std(), np.sqrt((s**2).mean()), np.abs(s).max()]
    return np.array(feats, dtype=float)

# ── LOCO-CV ──────────────────────────────────────────────────────────────
MODELS = {
    'Ridge': Ridge(alpha=1.0),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGB':   XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4,
                          subsample=0.8, random_state=42, verbosity=0),
    'SVR':   SVR(kernel='rbf', C=10, epsilon=0.05),
}

results_es = {m: [] for m in MODELS}
t1 = time.time()

print(f"\n{'Ratio':>6}", end='')
for m in MODELS: print(f"  {m:>8}", end='')
print()
print("-" * 50)

for ratio in RATIOS:
    X_list, y_list, cas_list = [], [], []
    for item in valid_passes:
        c      = item['case']
        segs   = all_segs_s[(c, item['run'])]
        segs_b = base_segs_s[c]
        f_raw  = extract_es(item['p'],        segs,   ratio)
        f_base = extract_es(base_run[c]['p'],  segs_b, ratio)
        meta   = np.array([item['DOC'], item['feed'], item['material']], dtype=float)
        X_list.append(np.concatenate([f_raw, f_raw - f_base, meta]))
        y_list.append(item['VB'])
        cas_list.append(c)

    X   = np.array(X_list)
    y   = np.array(y_list)
    cas = np.array(cas_list)

    print(f"  {ratio:.1f} ", end='', flush=True)
    for mname, mproto in MODELS.items():
        rmses = []
        for tc in CASES:
            tr = cas != tc; te = cas == tc
            sc  = StandardScaler()
            Xtr = sc.fit_transform(X[tr]); Xte = sc.transform(X[te])
            m   = copy.deepcopy(mproto)
            m.fit(Xtr, y[tr])
            rmses.append(np.sqrt(mean_squared_error(y[te], m.predict(Xte))))
        r = float(np.mean(rmses))
        results_es[mname].append(r)
        print(f"  {r:>8.4f}", end='', flush=True)
    print()

print(f"\n총 소요: {time.time()-t1:.1f}s")

print("\n=== 모델별 최적 ratio (Entry+Steady) ===")
for mname, vals in results_es.items():
    best_i = int(np.argmin(vals))
    print(f"  {mname:<6}: ratio={RATIOS[best_i]:.1f}  RMSE={vals[best_i]:.4f}")

# ── 시각화 ──────────────────────────────────────────────────────────────────
xgb_full   = {0.1:0.1851, 0.2:0.1724, 0.3:0.1300, 0.4:0.1242, 0.5:0.1172,
              0.6:0.1146, 0.7:0.1157, 0.8:0.1190, 0.9:0.1161, 1.0:0.1153}
xgb_steady = [0.1367, 0.1447, 0.1344, 0.1143, 0.1198, 0.1216, 0.1232, 0.1235, 0.1164, 0.1211]

x_pct = [r * 100 for r in RATIOS]
clr   = {'Ridge':'#9e9e9e', 'RF':'#43a047', 'XGB':'#e65100', 'SVR':'#1565c0'}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for mname, vals in results_es.items():
    best_i = int(np.argmin(vals))
    ax.plot(x_pct, vals, '-o', ms=5, lw=1.8, color=clr[mname],
            label=f'{mname} E+S (best={vals[best_i]:.4f}@{int(RATIOS[best_i]*100)}%)')
ax.plot(x_pct, [xgb_full[r] for r in RATIOS], 'k--s', ms=5, lw=1.5,
        label='XGB Full-signal best=0.1146@60%')
ax.plot(x_pct, xgb_steady, 'r:^', ms=5, lw=1.5, alpha=0.7,
        label='XGB Steady-only best=0.1143@40%')
ax.axhline(0.1146, color='black', ls=':', lw=1, alpha=0.4)
ax.set_xlabel('Entry+Steady 구간 사용 비율 (%)'); ax.set_ylabel('LOCO-CV RMSE (mm)')
ax.set_title(f'Entry+Steady Ratio  (W={W}/STEP={STEP}/DELTA={DELTA})')
ax.set_xticks(x_pct); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax2 = axes[1]
xgb_es  = results_es['XGB']
gaps    = [n - xgb_full[r] for n, r in zip(xgb_es, RATIOS)]
cols    = ['#43a047' if g <= 0 else '#ef5350' for g in gaps]
ax2.bar(x_pct, gaps, color=cols, edgecolor='black', lw=0.6)
ax2.axhline(0, color='black', lw=1)
ax2.set_xlabel('Entry+Steady 구간 사용 비율 (%)')
ax2.set_ylabel('DELTA_RMSE (E+S XGB - Full XGB)')
ax2.set_title('XGB: Entry+Steady vs 전체신호\n(음수 = E+S가 유리)')
ax2.set_xticks(x_pct); ax2.grid(axis='y', alpha=0.3)
for xp, g in zip(x_pct, gaps):
    sign = 'v' if g <= 0 else '^'
    va   = 'bottom' if g >= 0 else 'top'
    off  = 0.0005 if g >= 0 else -0.0005
    ax2.text(xp, g + off, f'{sign}{abs(g):.4f}', ha='center', va=va, fontsize=7.5)

plt.suptitle(f'Entry+Steady Input Ratio  (W={W}/STEP={STEP}/DELTA={DELTA}, Raw+Delta+Meta, LOCO-CV)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = '/content/drive/MyDrive/dataset/nasa_milling/entry_steady_ratio.png'
plt.savefig(save_path, dpi=150); plt.show()
print(f"\n저장: {save_path}")

# ── 수치 테이블 (3가지 조건 비교) ────────────────────────────────────────
print(f"\n{'Ratio':>6}  {'XGB(E+S)':>9}  {'XGB(St)':>9}  {'XGB(Full)':>10}  "
      f"{'D(E+S-Full)':>12}  {'D(E+S-St)':>10}")
print("-" * 65)
for i, ratio in enumerate(RATIOS):
    d1   = results_es['XGB'][i] - xgb_full[ratio]
    d2   = results_es['XGB'][i] - xgb_steady[i]
    s1   = 'v' if d1 <= 0 else '^'
    s2   = 'v' if d2 <= 0 else '^'
    print(f"  {ratio:.1f}   {results_es['XGB'][i]:>9.4f}  {xgb_steady[i]:>9.4f}"
          f"  {xgb_full[ratio]:>10.4f}  "
          f"{s1}{abs(d1):.4f}{'':>6}  {s2}{abs(d2):.4f}")

all_segs_s 재사용 (164개 패스, W=50/STEP=25/DELTA=1.0)
Entry+Steady 길이: mean=6654  std=1241  min=4000  max=8750

 Ratio     Ridge        RF       XGB       SVR
--------------------------------------------------
  0.1     0.2711    0.1851    0.1649    0.1994
  0.2     0.1763    0.1624    0.1605    0.1780
  0.3     0.1684    0.1406    0.1373    0.1628
  0.4     0.1520    0.1307    0.1293    0.1423
  0.5     0.1574    0.1253    0.1320    0.1391
  0.6     0.1613    0.1205    0.1171    0.1307
  0.7     0.1703    0.1218    0.1281    0.1289
  0.8     0.1505    0.1243    0.1251    0.1285
  0.9     0.1579    0.1193    0.1191    0.1210
  1.0     0.1652    0.1181    0.1149    0.1206

총 소요: 182.6s

=== 모델별 최적 ratio (Entry+Steady) ===
  Ridge : ratio=0.8  RMSE=0.1505
  RF    : ratio=1.0  RMSE=0.1181
  XGB   : ratio=1.0  RMSE=0.1149
  SVR   : ratio=1.0  RMSE=0.1206

저장: /content/drive/MyDrive/dataset/nasa_milling/entry_steady_ratio.png

 Ratio   XGB(E+S)    XGB(St)   XGB(Full)   D(E+S-Full)   D(E+S-St)
--

In [ ]:
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
